<a href="https://colab.research.google.com/github/mirdbg/Entrega_RAG/blob/main/S1_Herramientas_y_Bucle_Alumno.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 1 · Herramientas y el bucle

**Prácticas de *LLMs aplicados a Finanzas* · MIAX · jueves 10 de septiembre**

## Antes de empezar: por qué no vamos a montar un RAG

Lo esperable, en un curso de LLMs sobre documentos, sería montar un RAG
clásico: trocear los informes, calcular sus *embeddings*, guardarlos en un
índice vectorial y, ante cada pregunta, recuperar los *k* fragmentos más
parecidos y metérselos al modelo en el prompt. Recuperar, y luego generar. Es
el patrón que describe el paper de 2020 que le puso nombre
([Lewis et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP
Tasks*](https://arxiv.org/abs/2005.11401)) y sigue siendo el punto de partida
razonable para casi cualquier sistema de este tipo.

Vamos a construir todas esas piezas. Lo que no vamos a hacer es dejar que sean
**la arquitectura**, y conviene decir por qué.

Un RAG clásico decide **de antemano** qué información necesita el modelo. La
recuperación ocurre siempre, una sola vez, antes de generar, con la pregunta
tal y como llegó y con una *k* fijada de antemano. Eso funciona mientras la
pregunta se parezca a un párrafo del corpus. Deja de funcionar en cuanto:

- **la respuesta no está en un sitio, sino en dos.** «¿Qué riesgos añadió
  Microsoft entre FY2024 y FY2025?» exige recuperar dos veces y comparar. Una
  sola pasada de recuperación devuelve fragmentos de los dos años mezclados y
  el modelo se los inventa a medias;
- **el dato exacto no está en la prosa, sino en una tabla estructurada.**
  Buscar «beneficio neto de Apple» por similitud semántica devuelve párrafos
  que *hablan* del beneficio neto. La cifra auditada está en el XBRL, y no
  hace falta buscarla: se consulta;
- **la pregunta se refiere a algo que no existe.** Un RAG plano siempre
  devuelve sus *k* fragmentos, aunque la compañía por la que preguntáis no
  esté en el corpus. Siempre recupera algo, y ese algo siempre parece una
  respuesta.

La alternativa no es tirar el retrieval: es **bajarlo de arquitectura a
herramienta**. En lugar de un *pipeline* fijo que recupera y luego genera,
un modelo con varias herramientas que decide sobre la marcha cuál usar,
cuántas veces y con qué consulta. La documentación de LangChain llama a lo
primero *2-Step RAG* y a lo segundo *Agentic RAG*, y compara los dos en
[docs.langchain.com/oss/python/langchain/retrieval](https://docs.langchain.com/oss/python/langchain/retrieval).

Y el retrieval sobrevive ahí dentro por dos razones que no tienen nada que ver
con lo que el modelo sea capaz de leer:

- **Coste.** El corpus son 649.119 tokens. Meterlo entero en cada pregunta es
  pagarlo entero en cada pregunta. Lo calculáis vosotros en §2.
- **Auditabilidad.** En un entorno regulado hay que poder señalar el párrafo
  del que sale la cifra. El contexto largo da la respuesta; el retrieval da el
  ancla.

*(RAG lo visteis en teoría el sábado 12. Esto es lo que se hace con ello.)*

## Qué construimos hoy

Un **agente investigador sobre informes 10-K de la SEC**. Al final de la
sesión tendrá cuatro herramientas y sabrá elegir entre ellas:

| Herramienta | Para qué |
| --- | --- |
| `list_available()` | Comprobar qué hay en el corpus antes de inventárselo |
| `get_xbrl_fact()` | La cifra exacta, tal y como la reportó la compañía |
| `search_filings()` | Buscar en el texto de los informes. Hoy es caja negra |
| `read_section()` | El texto completo de una sección. Cara |

Esas cuatro herramientas son las tres carencias de arriba, resueltas. El
agente puede **recuperar dos veces** y comparar, porque quien decide cuántas
búsquedas hacen falta es él y no el *pipeline*. Puede **no buscar**, y
consultar la tabla XBRL cuando lo que se le pide es una cifra exacta. Y puede
**comprobar que algo existe** antes de responder, en lugar de devolver los
cinco fragmentos más parecidos a una pregunta sobre una empresa que no está.

El precio de esa flexibilidad es que el sistema se vuelve menos predecible: un
*pipeline* fijo siempre hace lo mismo, y un agente no. Por eso la sesión que
viene va entera de medirlo —evaluación de trayectoria, no solo de respuesta— y
de ponerle límites. Hoy toca que funcione; el día 17, que aguante.

Fijaos en la asimetría que hay en la tabla, porque es el eje de todo el curso:
`get_xbrl_fact` es barata, exacta y determinista, y `search_filings` es cara,
difusa y aproximada. **Elegir bien entre las dos es el trabajo del agente**, y
es lo que se evalúa. Un agente que acierta la cifra leyéndola de la prosa está
mal aunque el número salga bien: la próxima vez, con otra tabla partida, saldrá
mal y nadie se enterará.

## Qué se entrega el 24

Repositorio, golden set de 20 preguntas vuestras con al menos 6 comparativas,
informe con la tabla *baseline* contra final, y una presentación de 8 minutos
en la que ejecutáis **10 preguntas ciegas** que no veis hasta ese día. Todo
está en el enunciado que tenéis en la mano.

## Un aviso sobre el orden del curso

Hoy vais a escribir un bucle ReAct y a recuperar texto de un corpus. **El
viernes 18 os explicarán ReAct y el paper; RAG lo visteis el sábado 12.** Es
deliberado: hoy construís, y la teoría llega después a ponerle nombre a algo
que ya habréis tocado con las manos.

## Cómo se usa este notebook

- **Se ejecuta de arriba abajo, sin saltos.** No hay estado oculto: si saltáis
  una celda, la siguiente falla.
- Cada sección termina con `assert`. Si pasan, podéis seguir.
- **Ninguna clave está escrita aquí dentro.** Se piden por `getpass`.
- Lo que puede fallar por red está en `try/except` y degrada a un modo sin esa
  función. Nada de esto debería parar la clase.

Junto al notebook necesitáis dos ficheros más: `miax_s1.py` y
`demo_traza.json`. Y los dos ZIP del corpus, que se montan en §1.

In [1]:
# El agente terminado, antes de construirlo. Cinco minutos y ninguna
# explicación: esto es el destino, no el camino.
#
# Reproduce una ejecución grabada. Todavía no hay nada instalado ni montado.
try:
    import miax_s1
    _demo = miax_s1.demo_apertura()
except Exception as e:
    print(f"No se pudo reproducir la demo ({type(e).__name__}: {e}).")
    print("Comprueba que 'miax_s1.py' y 'demo_traza.json' están junto al "
          "notebook, en la misma carpeta.")

No se pudo reproducir la demo (ModuleNotFoundError: No module named 'miax_s1').
Comprueba que 'miax_s1.py' y 'demo_traza.json' están junto al notebook, en la misma carpeta.


In [2]:
# Instalación. Una sola celda, versiones fijadas, salida silenciada.
# Tarda alrededor de minuto y medio: mientras corre, leed la celda siguiente.
%pip install -q \
  langchain==1.3.18 langchain-core==1.6.1 langgraph==1.2.11 \
  langchain-openrouter==0.2.8 langchain-huggingface==1.2.2 \
  sentence-transformers==6.0.1 faiss-cpu==1.15.0 rank-bm25==0.2.2
print("Instalación terminada.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.3/837.3 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 56.3 MB/s eta 0:00:00
Instalación terminada.


In [3]:
%pip install -q langchain-google-genai langchain-groq langchain-cerebras

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 562.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 10.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


## Por qué van fijadas las versiones

Porque LangChain publica cada pocos días y una API que se mueve debajo de un
notebook lo rompe sin tocar una línea de código.

No es una precaución teórica: **los notebooks de la edición anterior de este
curso ya no ejecutan.** `create_react_agent`, `MemorySaver`,
`with_structured_output` y `RunnableWithMessageHistory` eran la API correcta
hace un año y hoy son otra cosa. Lo que veis aquí está verificado contra la
documentación oficial el 2 de septiembre de 2026.

De ahí salen dos hábitos que valen para cualquier proyecto que dependa de un
proveedor de modelos:

1. **Fijad la versión exacta**, no el rango. `>=1.3` no es una versión.
2. **Anotad la fecha de verificación** al lado del pin, para saber cómo de
   viejo es lo que estáis leyendo.

In [ ]:
# Claves. Nunca escritas en el notebook: se leen del entorno, y si no están,
# se piden por teclado sin que queden en la salida de la celda.
import os
import getpass


def pedir_clave(nombre: str, donde: str) -> bool:
    """Deja `nombre` en el entorno si se puede. Devuelve si hay clave."""
    if os.environ.get(nombre):
        print(f"{nombre}: ya estaba en el entorno.")
        return True
    try:
        valor = getpass.getpass(f"{nombre} (se saca en {donde}): ").strip()
    except Exception:                      # sin terminal interactiva
        valor = ""
    if valor:
        os.environ[nombre] = valor
        print(f"{nombre}: guardada en el entorno de esta sesión.")
        return True
    print(f"{nombre}: sin clave. Las celdas que llaman al modelo no van a "
          f"funcionar, pero las de datos sí.")
    return False


HAY_CLAVE = pedir_clave("GOOGLE_API_KEY", "aistudio.google.com/apikey")
HAY_CLAVE_GROQ = pedir_clave("GROQ_API_KEY", "console.groq.com/keys")
HAY_CLAVE_CEREBRAS = pedir_clave("CEREBRAS_API_KEY", "cloud.cerebras.ai")

# Una sola clave. La observabilidad de este curso no necesita ninguna más:
# la trayectoria se imprime en el propio notebook (§6).

GOOGLE_API_KEY (se saca en aistudio.google.com/apikey): ··········
GOOGLE_API_KEY: guardada en el entorno de esta sesión.
GROQ_API_KEY (se saca en console.groq.com/keys): ··········
GROQ_API_KEY: sin clave. Las celdas que llaman al modelo no van a funcionar, pero las de datos sí.
CEREBRAS_API_KEY (se saca en cloud.cerebras.ai): ··········
CEREBRAS_API_KEY: sin clave. Las celdas que llaman al modelo no van a funcionar, pero las de datos sí.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil, pathlib

origen = pathlib.Path("/content/drive/MyDrive/MIAX_Taller_NLP")
destino = pathlib.Path("/content/drive/MyDrive/MIAX_2026")
destino.mkdir(exist_ok=True)

for f in ["corpus_miax_2026.zip", "indice_faiss.zip", "miax_s1.py",
          "demo_traza.json", "golden_set_ejemplo.jsonl"]:
    if not (destino / f).exists():
        shutil.copy(origen / f, destino / f)

carpeta_deep = pathlib.Path("/content/_s1/pkg/deep")
carpeta_deep.mkdir(parents=True, exist_ok=True)
shutil.copy(destino / "miax_s1.py", carpeta_deep / "miax_s1.py")
shutil.copy(destino / "demo_traza.json", "/content/demo_traza.json")
shutil.copy(destino / "golden_set_ejemplo.jsonl", "/content/golden_set_ejemplo.jsonl")

import sys
sys.path.insert(0, str(carpeta_deep))

print("Todo listo: corpus/índice en MIAX_2026, miax_s1.py en carpeta profunda, "
      "demo_traza.json y golden_set_ejemplo.jsonl en /content.")

Todo listo: corpus/índice en MIAX_2026, miax_s1.py en carpeta profunda, demo_traza.json y golden_set_ejemplo.jsonl en /content.


In [ ]:
# %% Corpus e indice  --------------------------------
# Los dos ZIP os los pasamos nosotros (Drive compartido, aula virtual o el
# panel de ficheros de Colab): son 5,6 MB entre los dos. Nada de descargar
# de EDGAR en vivo, que con treinta cuadernos a la vez acaba en bloqueo.
#
# Si los teneis en Drive:
#     from google.colab import drive; drive.mount("/content/drive")
# y anadid la carpeta a CANDIDATOS.
import hashlib, pathlib, zipfile

PAQUETES = [
    ("corpus_miax_2026.zip", "4233c37fc9e9d12091af7a146063ad70903a3fe51404a485854f4021c63daee4"),
    ("indice_faiss.zip", "6b5610ad8ac6ea50364445d39bb464d993cbd87048fb07c4fe16657d7ac11655"),
]
URL_RESPALDO = ""          # vacio si no estan alojados
DESTINO = pathlib.Path("corpus")

CANDIDATOS = [
    pathlib.Path("."),
    pathlib.Path("/content"),
    pathlib.Path("/content/drive/MyDrive/MIAX_2026"),
    pathlib.Path("/content/drive/Shareddrives/MIAX_2026"),
]


def _sha256(ruta):
    d = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            d.update(b)
    return d.hexdigest()


def _localizar(nombre):
    for base in CANDIDATOS:
        ruta = base / nombre
        if ruta.is_file():
            return ruta
    if URL_RESPALDO:
        import urllib.request
        destino = pathlib.Path(nombre)
        urllib.request.urlretrieve(f"{URL_RESPALDO}/{nombre}", destino)
        return destino
    return None


try:
    for nombre, esperado in PAQUETES:
        origen = _localizar(nombre)
        assert origen is not None, (
            f"No encuentro {nombre}. Subelo con el panel de ficheros de "
            f"Colab (icono de carpeta a la izquierda), o monta el Drive "
            f"donde este. Buscado en: {[str(c) for c in CANDIDATOS]}"
        )
        obtenido = _sha256(origen)
        assert obtenido == esperado, (
            f"{nombre} no coincide con lo esperado: el fichero esta "
            f"corrupto o es de otra version.\n"
            f"  esperado: {esperado}\n  obtenido: {obtenido}"
        )
        with zipfile.ZipFile(origen) as zf:
            zf.extractall(DESTINO)

    # Los dos manifiestos declaran el hash de chunks.jsonl. El indice se
    # construyo sobre ESE fichero: si no cuadra, el indice y sus metadatos
    # estan desalineados y el retrieval devuelve el texto equivocado sin
    # dar ningun error.
    huella = _sha256(DESTINO / "chunks.jsonl")
    for manifiesto in ("MANIFEST.md", "indice/MANIFEST.md"):
        ruta = DESTINO / manifiesto
        if ruta.exists():
            assert huella in ruta.read_text(encoding="utf-8"), (
                f"chunks.jsonl no cuadra con {manifiesto}: el indice se "
                "construyo sobre otros fragmentos."
            )

    print("Corpus e indice verificados en", DESTINO.resolve())
    for p in sorted(DESTINO.rglob("*")):
        if p.is_file():
            rel = str(p.relative_to(DESTINO))
            print(f"  {rel:28s} {p.stat().st_size / 1e6:7.2f} MB")

except Exception as e:
    print("No se pudo preparar el corpus:", e)
    print("Pide los ficheros al profesor y dejalos junto al notebook.")

Corpus e indice verificados en /content/corpus
  LEEME.md                        0.00 MB
  MANIFEST.md                     0.00 MB
  chunks.jsonl                    3.80 MB
  indice/MANIFEST.md              0.00 MB
  indice/chunks_meta.parquet      1.48 MB
  indice/corpus.faiss             2.69 MB
  secciones.jsonl                 3.21 MB
  xbrl_facts.parquet              0.01 MB


In [ ]:
# Un proveedor es una cadena de texto.
#
# `init_chat_model` devuelve el mismo objeto sea cual sea el proveedor: el
# resto del notebook no se entera de cuál hay debajo. Cambiad la cadena y
# todo lo demás sigue igual.
from langchain.chat_models import init_chat_model

MODELO = "google_genai:gemini-3.8-flash"

modelo = None
if HAY_CLAVE:
    try:
        modelo = init_chat_model(MODELO, temperature=0)
        print(modelo.invoke("Responde solo con la palabra: listo").text)
    except Exception as e:
        print(f"No se pudo crear el modelo ({type(e).__name__}: {e}).")

# El mismo código contra otros cuatro proveedores. No los ejecutamos: cuestan
# dinero y hacen falta cuatro claves. El punto es que solo cambia la cadena.
#
#   init_chat_model("openrouter:anthropic/claude-opus-5", temperature=0)
#   init_chat_model("anthropic:claude-opus-5",            temperature=0)
#   init_chat_model("google_genai:gemini-3.8-flash",      temperature=0)
#   init_chat_model("openrouter:auto",                    temperature=0)
#
# El último deja que OpenRouter elija modelo. Sirve para enseñar el concepto y
# NO sirve para evaluar: si el modelo cambia entre dos ejecuciones, la
# comparación baseline-contra-final no significa nada.
#
# `temperature=0` va en todo lo que se vaya a evaluar. Con temperatura alta,
# dos ejecuciones de la misma pregunta dan métricas distintas y no sabéis si
# mejorasteis el sistema o tuvisteis suerte.

# --- verificación de §1 --------------------------------------------------
import pathlib
assert pathlib.Path("corpus/chunks.jsonl").is_file(), \
    "El corpus no está montado: repasad la celda de setup."
assert pathlib.Path("corpus/indice/corpus.faiss").is_file(), \
    "Falta el índice FAISS: descomprimid también indice_faiss.zip."
print("§1 listo.")

listo
§1 listo.


## NOTA: cascada de modelos, usada durante el desarrollo y retirada para el baseline

Mientras trabajamos con el free tier de Gemini (5-20 peticiones/minuto o
día según el modelo, insuficiente para iterar con comodidad — y el
crédito de $300 de Google Cloud no aplica a esta API desde marzo de
2026), usamos una cascada con `.with_fallbacks()`: si Gemini fallaba por
cuota, reintentaba con Groq y, si también fallaba, con Cerebras.

Activamos el pago (Prepay) en la API de Gemini antes de generar el
baseline, así que **retiramos la cascada**: el baseline y todas las
evaluaciones posteriores usan un único modelo fijo
(`gemini-3.8-flash`), sin fallback. Es necesario para que la comparación
baseline-vs-final sea válida — con la cascada, distintas preguntas
podrían haber sido respondidas por modelos distintos, lo que invalidaría
la comparación.

In [ ]:
from langchain.chat_models import init_chat_model

MODELO = "google_genai:gemini-3.8-flash"
modelo = init_chat_model(MODELO, temperature=0)

respuesta = modelo.invoke("Responde solo con la palabra: listo")
print(respuesta.content)

[{'type': 'text', 'text': 'listo', 'extras': {'signature': 'ErQECrEEAWkUfRNfBOC6hEBns/w69HOYzweKfOZV4YTlU/vzokZHk2TiSEHMnoQDQ4qDTthb34GHhQepHsc08U1gyo0xSDfgCGXIPPFHgM8W/Ar5Bz9kqc5Cr+1bEvdBkshaXMSP2deHZHewJWKuAyjKe+YhLtZDTQzR69I84SszBJAVY5oaq7Co/Pk+HEq4mk/IcfHwbJ107q9DNiC1+2o15qfVv40hB3/xAm3Q0AAES2BofGXK/dhBkr44aSUAOCQbGXHLgDixOpCS+YWnXWRoeN/77snH5vLBrkyJ+2SDAYmvO81Ifk8VARvPKxw/QVuHo2tRvrZD780rv+CUeNXTObJ0E6pX/3pjcB8YXPcRu7ZAGMFcwkzueX1GpfFy47MPsra4pu/zvbnNEMSqDwG8KiOQifsKZdbNV3utYzzmHpjRFAdQm/9+8qQFY/8Ke15OM9CdjZaT9wNVxK4DH6szMvmyUgYJXWjWP4v36niM5aGObOtZVD1o1fmVYHstmEDIG26ROuzxmLTi5DXZdVfpKpnlRXITOst0tXlV7pLKBgwcGrK2Ka8P8eG3effMBbtEYAK7Rj3Tqzo1DUixTzxmGYw32idM0lIl+67UzNFY31qGLoRa1iLfobi4kBPEv5u0OBDF7S9qSjbiAI3qmHFEh9NDjtKxNED0KOWanG1h/dzWxDNTdZGJuoUQGgaBXKfJiyj7CugsXu+3TLjCjTgKBahrKfgbEqlWm5LcsW1+EzmNb5Pc'}}]


## Anatomía de un 10-K

El 10-K es el informe anual que toda empresa cotizada en EE. UU. presenta ante
la SEC. Es un documento normalizado: los mismos epígrafes, en el mismo orden,
todos los años y en todas las compañías. Eso es lo que lo hace utilizable como
corpus.

Nos quedamos con cuatro epígrafes, que son donde está lo que se puede
preguntar:

| Item | Qué contiene | Qué se le pregunta |
| --- | --- | --- |
| **1A** · Risk Factors | Los riesgos que la compañía declara | Qué riesgos nuevos aparecen, cómo cambian entre ejercicios |
| **7** · MD&A | La dirección explicando sus propios resultados | Por qué subió o bajó una magnitud |
| **7A** · Market Risk | Exposición a tipos, divisa y precios | Cuantitativo y corto |
| **8** · Financial Statements | Los estados financieros y sus notas | Cifras, y de dónde salen |

Dos cosas que hay que saber del corpus antes de tocarlo:

**`fiscal_year` no es el año de presentación.** Las seis compañías cierran
ejercicio en cuatro meses distintos —NVDA en enero, MSFT en junio, AAPL en
septiembre, y GOOGL, META y AMZN en diciembre—, y está elegido así a
propósito. El 10-K de NVDA FY2025 se presentó en febrero de 2025; el de
Alphabet FY2025, en febrero de **2026**. Quien razone por fecha de
presentación se equivoca.

**NVIDIA no pone sus estados financieros bajo el Item 8.** Los deja bajo el
Item 15 y en el 8 escribe una remisión de dos líneas. El corpus sirve el
contenido correcto bajo la clave `"8"` y deja constancia en el campo
`item_origen`. Si no lo hiciera, `read_section("NVDA", 2025, "8")` devolvería
cuarenta tokens inútiles.

In [ ]:
# Cuánto ocupa un 10-K. Los tokens vienen precalculados en el corpus: contar
# en vivo tardaría más que la clase entera.
import json
import pandas as pd

secciones = pd.DataFrame(
    json.loads(l) for l in open("corpus/secciones.jsonl", encoding="utf-8")
)

tabla = secciones.pivot_table(
    index=["ticker", "fiscal_year"], columns="item", values="n_tokens"
).astype(int)
tabla["TOTAL"] = tabla.sum(axis=1)
print(tabla.to_string())

print(f"\nCorpus entero: {secciones.n_tokens.sum():,} tokens en "
      f"{len(secciones)} secciones")
print(f"Informe medio: {tabla['TOTAL'].mean():,.0f} tokens")
print(f"Informe mayor: {tabla['TOTAL'].max():,} · menor: "
      f"{tabla['TOTAL'].min():,}")

mayor = secciones.nlargest(1, "n_tokens").iloc[0]
# Ojo con `mayor.item`: en pandas eso es el método Series.item, no la
# columna. Con una columna que se llama 'item' hay que usar corchetes.
print(f"Sección mayor: {mayor['ticker']} FY{mayor['fiscal_year']} "
      f"Item {mayor['item']} con {mayor['n_tokens']:,} tokens")

item                   1A      7    7A      8  TOTAL
ticker fiscal_year                                  
AAPL   2024         11663   3814   612  15999  32088
       2025         11626   4294   612  16358  32890
AMZN   2024         10318   9597  1614  28097  49626
       2025         10516   9034  1547  29103  50200
GOOGL  2024         14727  11947  1877  30380  58931
       2025         14984  10648  1579  31845  59056
META   2024         33573  12621  1128  28501  75823
       2025         34751  12518  1144  32351  80764
MSFT   2024         12650  10295   409  28455  51809
       2025         11793   9510   409  26500  48212
NVDA   2024         18681   8348   635  26909  54573
       2025         19476   7824   638  27209  55147

Corpus entero: 649,119 tokens en 48 secciones
Informe medio: 54,093 tokens
Informe mayor: 80,764 · menor: 32,088
Sección mayor: META FY2025 Item 1A con 34,751 tokens


In [ ]:
# Los tres caminos hacia el mismo dato, con su precio.
PRECIOS_OPENROUTER = {
    # USD por millón de tokens (entrada, salida).
    # Consultado el 2/09/2026 en https://openrouter.ai/api/v1/models
    # REVISAR LA VÍSPERA: OpenRouter cambia precios sin avisar.
    "google/gemini-3.5-flash-lite":  (0.30,  2.50),
    "google/gemini-3.8-flash":       (0.75,  3.75),
    "anthropic/claude-opus-5":       (5.00, 25.00),
    "anthropic/claude-fable-5.1":   (10.00, 50.00),
}


def coste(entrada: int, salida: int, modelo: str) -> float:
    """Coste en dólares de una llamada, dados los tokens de cada lado."""
    p_in, p_out = PRECIOS_OPENROUTER[modelo]
    return entrada / 1e6 * p_in + salida / 1e6 * p_out


MODELO_COSTE = "google/gemini-3.8-flash"
SALIDA_TIPICA = 300      # tokens de respuesta, aproximadamente

# Tomamos una pregunta real: algo sobre el 10-K de Microsoft de FY2025.
informe = int(
    secciones[(secciones.ticker == "MSFT")
              & (secciones.fiscal_year == 2025)].n_tokens.sum()
)
fragmentos = 5 * 402     # cinco fragmentos, de unos 402 tokens de media
consulta_xbrl = 40       # ticker, ejercicio, concepto y el valor devuelto

caminos = [
    ("Informe entero en contexto", informe),
    ("Cinco fragmentos recuperados", fragmentos),
    ("Una consulta a XBRL", consulta_xbrl),
]

print(f"Modelo: {MODELO_COSTE}  "
      f"({PRECIOS_OPENROUTER[MODELO_COSTE][0]} $/M entrada, "
      f"{PRECIOS_OPENROUTER[MODELO_COSTE][1]} $/M salida)\n")
print("Camino                             entrada   $ entrada    $ total")
for nombre, tokens in caminos:
    solo_entrada = coste(tokens, 0, MODELO_COSTE)
    total = coste(tokens, SALIDA_TIPICA, MODELO_COSTE)
    print(f"{nombre:32s} {tokens:9,} {solo_entrada:11.5f} "
          f"{total:10.5f}")

# Dos lecturas distintas de la misma tabla, y las dos hacen falta:
caro, barato = caminos[0][1], caminos[2][1]
print(f"\nEn tokens de ENTRADA, el camino caro cuesta {caro / barato:,.0f} "
      f"veces más que la consulta a XBRL.")
print(f"Por pregunta completa, con {SALIDA_TIPICA} tokens de respuesta, "
      f"cuesta {coste(caro, SALIDA_TIPICA, MODELO_COSTE)
                / coste(barato, SALIDA_TIPICA, MODELO_COSTE):.0f} veces más: "
      f"la respuesta se paga igual por los tres caminos.")

# Y esa diferencia se multiplica por el número de preguntas y de grupos.
PREGUNTAS, GRUPOS = 40, 10
for nombre, tokens in caminos:
    c = coste(tokens, SALIDA_TIPICA, MODELO_COSTE) * PREGUNTAS * GRUPOS
    print(f"  {PREGUNTAS} preguntas x {GRUPOS} grupos, {nombre.lower()}: "
          f"{c:.2f} $")

# Y ahora la pregunta que de verdad importa, la comparativa entre dos
# ejercicios: hay que traerse los dos informes.
comparativa = int(
    secciones[(secciones.ticker == "MSFT")
              & (secciones.fiscal_year.isin([2024, 2025]))].n_tokens.sum()
)
todo = int(secciones.n_tokens.sum())
print(f"\nComparar FY2024 con FY2025 metiendo los dos informes enteros: "
      f"{comparativa:,} tokens = "
      f"{coste(comparativa, SALIDA_TIPICA, MODELO_COSTE):.4f} $")
print(f"Meter el corpus entero en cada pregunta: {todo:,} tokens = "
      f"{coste(todo, SALIDA_TIPICA, MODELO_COSTE):.4f} $")

# --- verificación de §2 --------------------------------------------------
assert len(secciones) == 48, "El corpus debería tener 48 secciones."
assert informe > fragmentos > consulta_xbrl
print("\n§2 listo.")

Modelo: google/gemini-3.8-flash  (0.75 $/M entrada, 3.75 $/M salida)

Camino                             entrada   $ entrada    $ total
Informe entero en contexto          48,212     0.03616    0.03728
Cinco fragmentos recuperados         2,010     0.00151    0.00263
Una consulta a XBRL                     40     0.00003    0.00115

En tokens de ENTRADA, el camino caro cuesta 1,205 veces más que la consulta a XBRL.
Por pregunta completa, con 300 tokens de respuesta, cuesta 32 veces más: la respuesta se paga igual por los tres caminos.
  40 preguntas x 10 grupos, informe entero en contexto: 14.91 $
  40 preguntas x 10 grupos, cinco fragmentos recuperados: 1.05 $
  40 preguntas x 10 grupos, una consulta a xbrl: 0.46 $

Comparar FY2024 con FY2025 metiendo los dos informes enteros: 100,021 tokens = 0.0761 $
Meter el corpus entero en cada pregunta: 649,119 tokens = 0.4880 $

§2 listo.


## La pregunta

Los tres caminos llevan al mismo dato, y la tabla dice dos cosas distintas
según por dónde se mire.

En **tokens de entrada**, meter el informe entero cuesta más de mil veces lo
que cuesta preguntarle a XBRL. Pero por **pregunta completa** la diferencia se
queda en unas treinta veces, porque los trescientos tokens de respuesta se
pagan igual por los tres caminos. Merece la pena fijarse en eso: a escala
pequeña, los costes fijos disimulan la diferencia. A escala de la práctica
—cuarenta preguntas por diez grupos, varias veces mientras iteráis— deja de
disimularla.

Y aun así el modelo **puede** leerse el informe entero: le cabe en la ventana
de contexto. Así que la pregunta no es si cabe.

> **¿Por qué seguir recuperando fragmentos, si el modelo puede leerlo todo?**

Pensadla un minuto antes de seguir. Hay al menos dos respuestas buenas y una
de ellas no tiene nada que ver con el dinero.

*(Y hay una tercera pregunta debajo: si tenemos el dato exacto en una tabla
XBRL, ¿por qué íbamos a buscarlo en la prosa? Eso es el §3.)*

## Qué es una herramienta, y qué ve el modelo de ella

Una *tool* es una función de Python que el modelo puede pedir que se ejecute.
El decorador `@tool` la convierte en un esquema —nombre, parámetros con sus
tipos, y descripción— y ese esquema viaja en la petición junto a los mensajes.

Lo que hay que entender es **qué parte de vuestra función ve el modelo**:

| Lo ve | No lo ve |
| --- | --- |
| El nombre de la función | El cuerpo |
| Los nombres y tipos de los parámetros | Los comentarios |
| El *docstring*, entero | Cómo de rápida o cara es |
| Lo que devuelve, cuando la llama | Lo que hace por dentro |

De ahí sale la consecuencia que gobierna el resto de la sesión: **el modelo
decide si os llama leyendo el docstring**. Un docstring vago produce un agente
que elige mal, con el mismo código debajo. Volveremos a esto en el segundo
ejercicio.

Empezamos por la herramienta fácil: exacta, determinista y prácticamente
gratis.

In [ ]:
from langchain.tools import tool

xbrl = pd.read_parquet("corpus/xbrl_facts.parquet")
print(f"{len(xbrl)} hechos XBRL · {xbrl.concept.nunique()} conceptos "
      f"distintos · {xbrl.ticker.nunique()} compañías")


@tool
def get_xbrl_fact(ticker: str, fiscal_year: int, concept: str) -> str:
    """Devuelve el valor EXACTO de una magnitud financiera tal y como la
    compañía la reportó en XBRL.

    Es la fuente autorizada para cualquier cifra. Úsala SIEMPRE en lugar de
    leer un número del texto del informe.

    Args:
        ticker: Símbolo bursátil, p. ej. 'NVDA'.
        fiscal_year: Ejercicio fiscal reportado, p. ej. 2024.
        concept: Concepto en taxonomía US-GAAP, p. ej. 'Revenues',
            'NetIncomeLoss', 'Assets', 'OperatingIncomeLoss'.

    Devuelve el valor con su unidad y fecha de cierre, o un aviso explícito
    si la compañía no reportó ese concepto en ese ejercicio.
    """
    filas = xbrl[(xbrl.ticker == ticker)
                 & (xbrl.fiscal_year == int(fiscal_year))
                 & (xbrl.concept == concept)]
    if filas.empty:
        disponibles = sorted(
            xbrl[(xbrl.ticker == ticker)
                 & (xbrl.fiscal_year == int(fiscal_year))].concept.unique()
        )
        if not disponibles:
            return (f"No hay datos de {ticker} para FY{fiscal_year} en el "
                    f"corpus. Usa list_available para ver qué hay.")
        return (f"{ticker} no reportó '{concept}' en FY{fiscal_year}. "
                f"Conceptos disponibles: {', '.join(disponibles)}")
    f = filas.iloc[0]
    return (f"{ticker} FY{fiscal_year} · {concept} = {f.value:,.0f} {f.unit} "
            f"(cierre de ejercicio {f.period_end}, según el {f.form})")


# Dos llamadas directas, sin modelo de por medio, para ver qué devuelve.
print(get_xbrl_fact.invoke(
    {"ticker": "NVDA", "fiscal_year": 2024, "concept": "Revenues"}))
print(get_xbrl_fact.invoke(
    {"ticker": "AMZN", "fiscal_year": 2025, "concept": "GrossProfit"}))

# La segunda no es un fallo del corpus: Amazon no etiqueta GrossProfit en
# us-gaap. Que la herramienta lo diga en vez de devolver vacío es la
# diferencia entre un agente que contesta "no está" y uno que se lo inventa.

135 hechos XBRL · 13 conceptos distintos · 6 compañías
NVDA FY2024 · Revenues = 60,922,000,000 USD (cierre de ejercicio 2024-01-28, según el 10-K)
AMZN no reportó 'GrossProfit' en FY2025. Conceptos disponibles: Assets, CashAndCashEquivalentsAtCarryingValue, EarningsPerShareBasic, EarningsPerShareDiluted, NetCashProvidedByUsedInOperatingActivities, NetIncomeLoss, OperatingIncomeLoss, RevenueFromContractWithCustomerExcludingAssessedTax, StockholdersEquity


In [ ]:
# La búsqueda en el texto de los informes.
#
# El cuerpo está en miax_s1.py y hoy es CAJA NEGRA a propósito: dentro hay
# troceado, embeddings, un índice FAISS y una decisión de top-k, y cada una de
# esas cuatro cosas se puede hacer mejor o peor. El día 17 se abre la caja, se
# mide lo que hace y se arregla tres veces.
#
# Hoy interesa otra cosa: que es la herramienta CARA y DIFUSA, la contraria de
# get_xbrl_fact. Devuelve texto que se parece a lo que pedisteis, no la
# respuesta.
import miax_s1


@tool
def search_filings(query: str, ticker: str | None = None,
                   fiscal_year: int | None = None,
                   item: str | None = None, k: int = 5) -> str:
    """Busca fragmentos de texto relevantes en los informes 10-K del corpus.

    Úsala para preguntas cualitativas: riesgos, estrategia, litigios,
    comentarios de la dirección. NO la uses para obtener cifras: para eso
    está get_xbrl_fact.

    Args:
        query: Qué buscar, en lenguaje natural.
        ticker: Filtra por compañía si la pregunta la menciona.
        fiscal_year: Filtra por ejercicio si la pregunta lo menciona.
        item: Filtra por sección: '1A' riesgos, '7' MD&A,
            '7A' riesgo de mercado, '8' estados financieros.
        k: Número de fragmentos a devolver.

    Devuelve k fragmentos, cada uno con su chunk_id para poder citarlo.
    """
    return miax_s1.formatear_fragmentos(
        miax_s1.buscar(query, ticker=ticker, fiscal_year=fiscal_year,
                       item=item, k=k)
    )


# La primera llamada tarda unos segundos: carga el modelo de embeddings.
salida = search_filings.invoke({
    "query": "risks from misuse of our AI systems by third parties",
    "ticker": "MSFT", "fiscal_year": 2025, "item": "1A", "k": 3,
})
print(salida[:1200], "...")

# Dos cosas que mirar en esa salida:
#
# El corpus está EN INGLÉS. La consulta también tiene que ir en inglés aunque
# la pregunta del usuario venga en español. Que el agente traduzca la consulta
# es parte de su trabajo.
#
# Y el fragmento empieza a mitad de frase ("of operations."). Nadie ha decidido
# que empiece ahí: es donde cayó el corte del troceador. Eso es exactamente lo
# que se abre y se arregla el día 17.

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[MSFT-2025-1A-0017] MSFT FY2025 Item 1A (similitud 0.842)
of operations.



Issues in the development, deployment, and use of AI may result in reputational or competitive harm or liability. We are building AI into many of our offerings, including our productivity services, and we are also making AI available for our customers to use in solutions that they build. This AI may be developed by Microsoft or others, including our strategic partner, OpenAI. We expect these elements of our business to grow. We envision a future in which AI operating in devices, applications, and the cloud helps our customers be more productive in their work and personal lives. As with many innovations, AI presents risks and challenges that could affect its adoption, and therefore our business. AI algorithms or training methodologies may be flawed. Datasets may be overbroad, insufficient, or contain biased or inaccurate information. Content generated by AI systems may be offensive, illegal, inaccurate, or other

In [ ]:
# La vía de contexto largo. Existe para que el agente pueda elegir pagarla.
@tool
def read_section(ticker: str, fiscal_year: int, item: str) -> str:
    """Devuelve el TEXTO COMPLETO de una sección de un 10-K.

    Es una herramienta CARA: puede devolver decenas de miles de tokens.
    Úsala solo cuando search_filings devuelva fragmentos insuficientes y
    necesites el contexto entero de una sección concreta.

    Args:
        ticker: Símbolo bursátil, p. ej. 'META'.
        fiscal_year: Ejercicio fiscal, p. ej. 2025.
        item: '1A' riesgos, '7' MD&A, '7A' riesgo de mercado,
            '8' estados financieros.
    """
    filas = secciones[(secciones.ticker == ticker)
                      & (secciones.fiscal_year == int(fiscal_year))
                      & (secciones.item == item)]
    if filas.empty:
        return (f"No hay Item {item} de {ticker} FY{fiscal_year} en el "
                f"corpus. Usa list_available para ver qué hay.")
    f = filas.iloc[0]
    return f.texto


# No la llamamos con el modelo delante: la sección más larga del corpus son
# 34.751 tokens y pagarlos para ver que funciona no tiene sentido. Miramos
# el tamaño de lo que devolvería.
for t, fy, it in [("AAPL", 2024, "7A"), ("META", 2025, "1A")]:
    texto = read_section.invoke(
        {"ticker": t, "fiscal_year": fy, "item": it})
    n = int(secciones[(secciones.ticker == t)
                      & (secciones.fiscal_year == fy)
                      & (secciones.item == it)].iloc[0].n_tokens)
    print(f"{t} FY{fy} Item {it}: {len(texto):,} caracteres, {n:,} tokens")

# Nada impide al agente llamar a read_section cuatro veces seguidas y
# quemarse el presupuesto del grupo en una pregunta. Hoy no hay nada que se
# lo impida; el día 17 se le pone un ToolCallLimitMiddleware.

AAPL FY2024 Item 7A: 3,043 caracteres, 612 tokens
META FY2025 Item 1A: 195,308 caracteres, 34,751 tokens


In [ ]:
# EJERCICIO 1 (10 min) · list_available()
#
# Sin esta herramienta el agente se inventa compañías y ejercicios que no
# están en el corpus: no tiene forma de comprobar el mundo, así que rellena
# el hueco con lo que le suena.
#
# Tiene que devolver, en un texto que el modelo pueda leer: qué tickers hay,
# el nombre de cada compañía, qué ejercicios y qué items. Todo sale de
# `secciones`, que ya está cargado (columnas: ticker, empresa, fiscal_year,
# item, ...).
#
# Dos avisos:
#   - El docstring es parte del ejercicio. Escribid uno que le diga al modelo
#     CUÁNDO llamarla, no solo qué hace.
#   - Devolved un texto, no un DataFrame: lo que devuelve la herramienta se
#     le pasa al modelo tal cual.
@tool
def list_available() -> str:
    """Lista qué compañías, ejercicios y secciones existen en el corpus.

    Úsala SIEMPRE antes de responder que un dato no existe, y antes de
    llamar a cualquier otra herramienta si no estás seguro de que la
    compañía o el ejercicio que te piden estén en el corpus.
    """
    lineas = []
    for (ticker, empresa), grupo in secciones.groupby(["ticker", "empresa"]):
        ejercicios = sorted(grupo["fiscal_year"].unique())
        for fy in ejercicios:
            items = sorted(grupo[grupo["fiscal_year"] == fy]["item"].unique())
            lineas.append(f"{ticker} ({empresa}) FY{fy}: items {', '.join(items)}")
    return "\n".join(lineas)


print(list_available.invoke({}))

AAPL (Apple Inc.) FY2024: items 1A, 7, 7A, 8
AAPL (Apple Inc.) FY2025: items 1A, 7, 7A, 8
AMZN (AMAZON COM INC) FY2024: items 1A, 7, 7A, 8
AMZN (AMAZON COM INC) FY2025: items 1A, 7, 7A, 8
GOOGL (Alphabet Inc.) FY2024: items 1A, 7, 7A, 8
GOOGL (Alphabet Inc.) FY2025: items 1A, 7, 7A, 8
META (Meta Platforms, Inc.) FY2024: items 1A, 7, 7A, 8
META (Meta Platforms, Inc.) FY2025: items 1A, 7, 7A, 8
MSFT (MICROSOFT CORP) FY2024: items 1A, 7, 7A, 8
MSFT (MICROSOFT CORP) FY2025: items 1A, 7, 7A, 8
NVDA (NVIDIA CORP) FY2024: items 1A, 7, 7A, 8
NVDA (NVIDIA CORP) FY2025: items 1A, 7, 7A, 8


In [ ]:
# El problema que resuelve: preguntar por algo que no está en el corpus.
#
# Tesla no está. La pregunta es qué hace el agente con eso.
PREGUNTA_FUERA = ("¿Cuál fue el revenue de Tesla en el ejercicio 2025 según "
                  "su 10-K?")

if modelo is not None:
    sin_lista = modelo.bind_tools([get_xbrl_fact, search_filings])
    con_lista = modelo.bind_tools([get_xbrl_fact, search_filings,
                                   list_available])

    for etiqueta, m in [("SIN list_available", sin_lista),
                        ("CON list_available", con_lista)]:
        r = m.invoke([{"role": "user", "content": PREGUNTA_FUERA}])
        print(f"\n--- {etiqueta} ---")
        if r.tool_calls:
            for tc in r.tool_calls:
                print(f"  llama a {tc['name']}({tc['args']})")
        else:
            print(f"  contesta directamente: {r.text[:200]}")
else:
    print("Sin clave: esta celda necesita el modelo. Seguid leyendo.")

# Lo que se ve casi siempre: sin la herramienta, el modelo llama a
# get_xbrl_fact con ticker='TSLA' y se queda esperando un dato que no existe,
# o directamente responde de memoria una cifra de Tesla que no ha salido de
# ningún informe. Con la herramienta, muchas veces comprueba primero.
#
# "Muchas veces" no es "siempre", y eso también es parte de la lección: una
# herramienta no es una garantía, es una opción que el modelo puede tomar.
# Convertirla en garantía es el trabajo del día 17 (guardrails).


--- SIN list_available ---
  llama a get_xbrl_fact({'concept': 'Revenues', 'ticker': 'TSLA', 'fiscal_year': 2025})

--- CON list_available ---
  llama a list_available({})


In [ ]:
# EJERCICIO 2 (10 min) · Dos docstrings, el mismo código
#
# Abajo está get_xbrl_fact_vago: cuerpo idéntico al de get_xbrl_fact, con una
# descripción que no dice nada. Es la clase de docstring que se escribe cuando
# uno piensa que el docstring es documentación.
@tool
def get_xbrl_fact_vago(ticker: str, fiscal_year: int, concept: str) -> str:
    """Devuelve un dato financiero."""
    return get_xbrl_fact.func(ticker, fiscal_year, concept)


PREGUNTA = "¿Cuál fue el beneficio neto de Apple en el ejercicio 2025?"

# TODO (alumno): hacerle la MISMA pregunta al modelo dos veces, cambiando
# solo cuál de las dos versiones de la herramienta está en la lista, e
# imprimir qué herramienta eligió cada vez y con qué argumentos.
#
# Pistas:
#   modelo.bind_tools([...]) devuelve un modelo que puede pedir herramientas.
#   .invoke([{"role": "user", "content": PREGUNTA}]) lo llama.
#   La respuesta trae .tool_calls, una lista de dicts con 'name' y 'args'.
#
# Y una pregunta para contestar en voz alta: además de QUÉ herramienta eligió,
# mirad con qué `concept` la llamó. ¿De dónde iba a sacar el nombre correcto
# con el docstring vago?
sin_docstring = modelo.bind_tools([get_xbrl_fact_vago, search_filings])
con_docstring = modelo.bind_tools([get_xbrl_fact, search_filings])

for etiqueta, m in [("DOCSTRING VAGO", sin_docstring),
                    ("DOCSTRING BUENO", con_docstring)]:
    r = m.invoke([{"role": "user", "content": PREGUNTA}])
    print(f"\n--- {etiqueta} ---")
    if r.tool_calls:
        for tc in r.tool_calls:
            print(f"  llama a {tc['name']}({tc['args']})")
    else:
        print(f"  contesta directamente: {r.text[:200]}")

# --- verificación de §3 --------------------------------------------------
assert "NVDA" in list_available.invoke({}), \
    "list_available debería nombrar las compañías del corpus."
assert "60,922,000,000" in get_xbrl_fact.invoke(
    {"ticker": "NVDA", "fiscal_year": 2024, "concept": "Revenues"})
assert "no reportó" in get_xbrl_fact.invoke(
    {"ticker": "AMZN", "fiscal_year": 2025, "concept": "GrossProfit"}), \
    "La herramienta debe decir explícitamente que el concepto no está."
print("§3 listo: cuatro herramientas definidas.")


--- DOCSTRING VAGO ---
  llama a get_xbrl_fact_vago({'ticker': 'AAPL', 'concept': 'NetIncomeLoss', 'fiscal_year': 2025})

--- DOCSTRING BUENO ---
  llama a get_xbrl_fact({'ticker': 'AAPL', 'concept': 'NetIncomeLoss', 'fiscal_year': 2025})
§3 listo: cuatro herramientas definidas.


In [ ]:
# PRUEBA EXTRA (nuestra, no del enunciado): forzar la diferencia con un
# concepto XBRL menos estándar, donde el nombre varía según la empresa.
PREGUNTA_EXTRA = "¿Cuáles fueron los ingresos totales de Amazon en el ejercicio 2025?"

for etiqueta, m in [("DOCSTRING VAGO", sin_docstring),
                    ("DOCSTRING BUENO", con_docstring)]:
    r = m.invoke([{"role": "user", "content": PREGUNTA_EXTRA}])
    print(f"\n--- {etiqueta} ---")
    if r.tool_calls:
        for tc in r.tool_calls:
            print(f"  llama a {tc['name']}({tc['args']})")
    else:
        print(f"  contesta directamente: {r.text[:200]}")


--- DOCSTRING VAGO ---
  llama a get_xbrl_fact_vago({'concept': 'Revenues', 'fiscal_year': 2025, 'ticker': 'AMZN'})

--- DOCSTRING BUENO ---
  llama a get_xbrl_fact({'fiscal_year': 2025, 'concept': 'Revenues', 'ticker': 'AMZN'})


In [ ]:
# PRUEBA EXTRA 2: un concepto donde el nombre XBRL real no es el que un
# humano (o un modelo sin pista) adivinaría de forma natural.
PREGUNTA_EXTRA2 = "¿Cuál fue el resultado operativo (operating income) de NVIDIA en el ejercicio 2024?"

for etiqueta, m in [("DOCSTRING VAGO", sin_docstring),
                    ("DOCSTRING BUENO", con_docstring)]:
    r = m.invoke([{"role": "user", "content": PREGUNTA_EXTRA2}])
    print(f"\n--- {etiqueta} ---")
    if r.tool_calls:
        for tc in r.tool_calls:
            print(f"  llama a {tc['name']}({tc['args']})")
    else:
        print(f"  contesta directamente: {r.text[:200]}")


--- DOCSTRING VAGO ---
  llama a get_xbrl_fact_vago({'ticker': 'NVDA', 'concept': 'OperatingIncomeLoss', 'fiscal_year': 2024})

--- DOCSTRING BUENO ---
  llama a get_xbrl_fact({'ticker': 'NVDA', 'fiscal_year': 2024, 'concept': 'OperatingIncomeLoss'})


## NOTA: el Ejercicio 2 no mostró diferencia con Gemini

Probamos tres conceptos distintos (`NetIncomeLoss`, `Revenues` para Amazon,
`OperatingIncomeLoss`) comparando `get_xbrl_fact` (docstring completo) contra
`get_xbrl_fact_vago` (`"Devuelve un dato financiero."`). En los tres casos,
**ambas versiones eligieron el mismo `concept`** — sin diferencia entre
docstring bueno y vago.

Interpretación: con un modelo como `gemini-3.8-flash`, el vocabulario de la
taxonomía US-GAAP (`Revenues`, `NetIncomeLoss`, `OperatingIncomeLoss`, etc.)
ya forma parte de su conocimiento general de entrenamiento, así que el
docstring no añade información que el modelo no tuviera ya. La lección del
ejercicio (*"el docstring es prompt engineering, no documentación"*) sigue
siendo válida en general, pero aquí no se hizo visible porque:

- Los conceptos probados son relativamente estándar en la taxonomía XBRL.
- No probamos con un vocabulario genuinamente inventado o específico del
  corpus, que el modelo no pudiera conocer de antemano.

La diferencia sí sería esperable con modelos menos capaces, o con conceptos
verdaderamente ambiguos o no estándar (p. ej., un vocabulario propio del
dominio que no exista en ningún dataset de entrenamiento público).

## La descripción de una *tool* es *prompt engineering*, no documentación

El código de `get_xbrl_fact` y el de `get_xbrl_fact_vago` es exactamente el
mismo. Lo único que cambia es un párrafo de texto, y con él cambia el
comportamiento del agente. No hay ninguna otra parte del sistema donde
reescribir un comentario altere lo que hace el programa.

Tres consecuencias prácticas, que valen para el entregable:

1. **Decid cuándo llamarla, no solo qué hace.** «Es la fuente autorizada para
   cualquier cifra; úsala SIEMPRE en lugar de leer un número del texto» es
   una instrucción de enrutado.
2. **Decid cuándo *no* llamarla.** El docstring de `search_filings` dice que
   no se use para cifras. Esa frase vale más que las otras cinco.
3. **Poned el vocabulario dentro.** Si el parámetro espera `'1A'`, `'7'`,
   `'7A'` u `'8'`, el docstring tiene que enumerarlos: el modelo no puede
   adivinar un vocabulario que no ha visto.

---

## Pausa · 10 minutos

Al volver escribimos a mano el bucle que hace funcionar todo esto.

## `bind_tools`: qué devuelve el modelo cuando quiere una herramienta

Un modelo de lenguaje no ejecuta nada. Lo único que sabe hacer es producir
texto. Cuando decimos que «llama a una herramienta», lo que ocurre es esto:

1. Le mandamos los mensajes **y los esquemas** de las herramientas.
2. Él responde con una petición estructurada: nombre, argumentos y un `id`.
3. **Nosotros** ejecutamos la función.
4. Le devolvemos el resultado en un mensaje de tipo `tool`, con el mismo `id`.
5. Vuelta a empezar hasta que responda sin pedir nada.

Ese bucle de cinco pasos es todo. Lo importante es dónde está la frontera: el
paso 3 lo hace vuestro código, no el modelo. Un agente es un bucle `while`
alrededor de una llamada a un modelo, y quien ejecuta sois vosotros.

`model.bind_tools([...])` es lo que mete los esquemas en la petición.

In [ ]:
# Qué devuelve exactamente el modelo cuando pide una herramienta.
HERRAMIENTAS = [list_available, get_xbrl_fact, search_filings, read_section]
POR_NOMBRE = {t.name: t for t in HERRAMIENTAS}

if modelo is not None:
    modelo_con_tools = modelo.bind_tools(HERRAMIENTAS)

    respuesta = modelo_con_tools.invoke([
        {"role": "user",
         "content": "¿Cuáles fueron los ingresos de NVIDIA en FY2025?"},
    ])

    print("texto de la respuesta:", repr(respuesta.text))
    print("\ntool_calls, en crudo:")
    for tc in respuesta.tool_calls:
        print(f"  name : {tc['name']}")
        print(f"  args : {tc['args']}")
        print(f"  id   : {tc['id']}")
else:
    modelo_con_tools = None
    print("Sin clave: esta celda necesita el modelo.")

texto de la respuesta: ''

tool_calls, en crudo:
  name : list_available
  args : {}
  id   : call_736817


In [ ]:
# EL BUCLE. Treinta líneas, sin framework. El andamiaje está puesto; el
# cuerpo del bucle interior es vuestro.
from langchain.messages import ToolMessage

SYSTEM = """Eres un analista financiero que responde preguntas sobre informes
10-K usando ÚNICAMENTE las herramientas disponibles.

Reglas:
- Para cualquier CIFRA, usa get_xbrl_fact. Nunca leas un número de la prosa.
- Para riesgos, estrategia o comentarios de la dirección, usa search_filings.
- Si no sabes si una compañía o un ejercicio están en el corpus, empieza por
  list_available.
- El corpus está en inglés: escribe las consultas de búsqueda en inglés.
- Cita el chunk_id del fragmento en el que te apoyes.
- Si el dato no está en el corpus, dilo. No lo estimes.
"""

modelo_con_tools = modelo.bind_tools(HERRAMIENTAS)


def agente_manual(pregunta: str, max_vueltas: int = 6,
                  verboso: bool = True) -> str:
    mensajes = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": pregunta}]
    for vuelta in range(max_vueltas):
        respuesta = modelo_con_tools.invoke(mensajes)
        mensajes.append(respuesta)

        if not respuesta.tool_calls:
            return respuesta.text

        for tc in respuesta.tool_calls:
            if verboso:
                print(f"vuelta {vuelta + 1}: {tc['name']}({tc['args']})")
            try:
                herramienta = POR_NOMBRE[tc["name"]]
                resultado = herramienta.invoke(tc["args"])
            except Exception as e:
                resultado = f"Error ejecutando {tc['name']}: {e}"
            mensajes.append(
                ToolMessage(content=str(resultado),
                            tool_call_id=tc["id"],
                            name=tc["name"])
            )
    return "Se agotaron las vueltas sin llegar a una respuesta."


print("agente_manual definido.")

agente_manual definido.


In [ ]:
# Una pregunta que no se contesta con una sola herramienta: hace falta el
# texto (qué riesgo nuevo) y la cifra (cuánto creció). Dos herramientas
# distintas, y el modelo tiene que decidir el orden.
PREGUNTA_DOBLE = (
    "¿Qué riesgos nuevos relacionados con la inteligencia artificial añadió "
    "Microsoft en su 10-K de FY2025 respecto al de FY2024, y cuánto creció su "
    "revenue entre esos dos ejercicios?"
)

if modelo_con_tools is not None:
    print("TRAYECTORIA")
    final = agente_manual(PREGUNTA_DOBLE, max_vueltas=8)
    print("\nRESPUESTA\n", final)
else:
    print("Sin clave: esta celda necesita el modelo.")

# Comparad la trayectoria con la de la demo del principio de la sesión.
# Es la misma, y acabáis de escribirla.

# --- verificación de §5 --------------------------------------------------
import inspect
assert "ToolMessage" in inspect.getsource(agente_manual), \
    "El bucle tiene que devolverle el resultado al modelo con un ToolMessage."
assert "tool_call_id" in inspect.getsource(agente_manual), \
    "Cada ToolMessage necesita el tool_call_id de su petición."
print("\n§5 listo.")

TRAYECTORIA
vuelta 1: list_available({})
vuelta 2: get_xbrl_fact({'concept': 'Revenues', 'fiscal_year': 2024, 'ticker': 'MSFT'})
vuelta 2: get_xbrl_fact({'fiscal_year': 2025, 'ticker': 'MSFT', 'concept': 'Revenues'})
vuelta 3: get_xbrl_fact({'fiscal_year': 2024, 'concept': 'RevenueFromContractWithCustomerExcludingAssessedTax', 'ticker': 'MSFT'})
vuelta 3: get_xbrl_fact({'ticker': 'MSFT', 'concept': 'RevenueFromContractWithCustomerExcludingAssessedTax', 'fiscal_year': 2025})
vuelta 4: search_filings({'fiscal_year': 2025, 'item': '1A', 'query': 'artificial intelligence AI risk', 'k': 10, 'ticker': 'MSFT'})
vuelta 5: search_filings({'ticker': 'MSFT', 'fiscal_year': 2024, 'item': '1A', 'k': 10, 'query': 'artificial intelligence AI risk'})
vuelta 6: search_filings({'k': 5, 'query': 'trade sanctions export controls AI Diffusion Rule', 'ticker': 'MSFT', 'fiscal_year': 2024, 'item': '1A'})
vuelta 7: search_filings({'query': 'agentic AI autonomous', 'ticker': 'MSFT', 'item': '1A', 'k': 5, 'fisc

In [ ]:
# EL BUCLE INFINITO. Esta celda está diseñada para portarse mal.
#
# La pregunta pide el margen bruto de Amazon. Amazon NO etiqueta GrossProfit
# en us-gaap, así que get_xbrl_fact devuelve un aviso una y otra vez, y el
# modelo tiende a reintentar con variantes del concepto en lugar de rendirse.
#
# Con max_vueltas=100 no hay nada que lo pare. Dejadlo correr unas cuantas
# vueltas para verlo y CORTAD LA CELDA A MANO (el cuadrado de stop).
#
# Ojo: cada vuelta cuesta dinero del presupuesto del grupo. Diez vueltas
# bastan para entenderlo.
PREGUNTA_BUCLE = ("Compara el margen bruto de Amazon en FY2024 y FY2025 y "
                  "explica a qué se debe el cambio.")

# TODO: cambiar a 100 para que vuelva a ser un bucle infinito
if modelo_con_tools is not None:
    print(agente_manual(PREGUNTA_BUCLE, max_vueltas=8))
else:
    print("Sin clave: esta celda necesita el modelo.")

# NO lo arreglamos hoy. El día 17 se cierra con una línea:
#     ToolCallLimitMiddleware(run_limit=8)
# El valor de este bloque está en haber visto el problema una semana antes de
# ver la solución.

vuelta 1: list_available({})
vuelta 2: get_xbrl_fact({'fiscal_year': 2024, 'concept': 'GrossProfit', 'ticker': 'AMZN'})
vuelta 3: search_filings({'ticker': 'AMZN', 'item': '8', 'k': 5, 'fiscal_year': 2025, 'query': 'Consolidated Statements of Operations Net sales Cost of sales'})
vuelta 4: get_xbrl_fact({'fiscal_year': 2024, 'ticker': 'AMZN', 'concept': 'RevenueFromContractWithCustomerExcludingAssessedTax'})
vuelta 5: get_xbrl_fact({'concept': 'RevenueFromContractWithCustomerExcludingAssessedTax', 'fiscal_year': 2025, 'ticker': 'AMZN'})
vuelta 6: get_xbrl_fact({'concept': 'CostOfGoodsAndServicesSold', 'ticker': 'AMZN', 'fiscal_year': 2025})
vuelta 7: search_filings({'item': '7', 'ticker': 'AMZN', 'query': '"gross margin" OR "gross profit" OR "Cost of sales"', 'k': 5, 'fiscal_year': 2025})
vuelta 8: search_filings({'query': '"gross margin" OR "margins" operational efficiencies Cost of sales', 'item': '7', 'k': 4, 'ticker': 'AMZN', 'fiscal_year': 2025})
Se agotaron las vueltas sin llegar

# NOTA

El agente **no encontró `GrossProfit`** en XBRL para Amazon (no lo reporta,
como avisa el propio corpus). Pero en vez de admitir que el dato no está
disponible, **calculó el margen bruto a mano**: sacó ventas netas y coste
de ventas del texto plano de la sección 8 vía `search_filings`, y los restó
él mismo.

Esto viola dos reglas explícitas del `SYSTEM` prompt:
- *"Para cualquier CIFRA, usa `get_xbrl_fact`. Nunca leas un número de la
  prosa"* — las cifras vinieron de texto, no de XBRL.
- *"Si el dato no está en el corpus, dilo. No lo estimes"* — el agente
  estimó un margen derivado en vez de decirlo.

El número final puede ser correcto, pero **por trayectoria es un fallo**,
exactamente el ejemplo que da el enunciado al principio: un agente que
acierta la cifra por el camino equivocado no cuenta como acierto. Es el
primer caso real que tenemos para la clasificación de errores del baseline
que hay que llevar a la Sesión 2.

## Lo que acabáis de escribir tiene nombre

Se llama **ReAct** (*Reasoning + Acting*), es de 2022, y es el patrón sobre el
que está construida la mayor parte de los agentes que hay hoy en producción.
El bucle de la celda anterior es, sin quitar ni añadir nada: razonar, actuar,
observar, repetir.

**El viernes 18 os lo van a explicar con el paper delante.** Hoy lo habéis
escrito. Cuando lleguéis a esa clase, no vais a estar aprendiendo un patrón
nuevo: vais a estar poniéndole nombre a las treinta líneas que ya tenéis.

Ese es también el motivo de que el bloque siguiente vaya después y no antes.
`create_agent` hace esto mismo en cinco líneas, y solo se aprecia lo que
resuelve si antes lo habéis escrito a mano.

In [ ]:
# El esquema de respuesta. Es el contrato §7 del enunciado, literal.
#
# Fijaos en lo que hace: convierte la cita de una súplica en el prompt
# ("por favor, cita la fuente") en un requisito estructural. El modelo no
# puede devolver una respuesta sin decir de dónde sale, porque el esquema no
# valida. Y los evaluadores del día 17 leen campos en lugar de parsear prosa.
from typing import Literal

from pydantic import BaseModel, Field


class RespuestaFinanciera(BaseModel):
    """Respuesta trazable a una pregunta sobre informes 10-K."""

    respuesta: str = Field(
        description="Respuesta en prosa, breve y directa")
    cifra: float | None = Field(
        default=None, description="Valor numérico, si la pregunta pide uno")
    unidad: str | None = Field(
        default=None, description="USD, shares, porcentaje…")
    ticker: str | None = None
    ejercicio: int | None = None
    fuente: Literal["xbrl", "texto", "ambas", "ninguna"] = Field(
        description="De dónde sale el dato. 'ninguna' si no está en el corpus")
    cita: str | None = Field(
        default=None,
        description="Texto literal del informe que respalda la respuesta")
    chunk_id: str | None = Field(
        default=None,
        description="Identificador del fragmento citado, para verificar")


print(json.dumps(RespuestaFinanciera.model_json_schema()["properties"],
                 indent=2, ensure_ascii=False)[:600], "...")

{
  "respuesta": {
    "description": "Respuesta en prosa, breve y directa",
    "title": "Respuesta",
    "type": "string"
  },
  "cifra": {
    "anyOf": [
      {
        "type": "number"
      },
      {
        "type": "null"
      }
    ],
    "default": null,
    "description": "Valor numérico, si la pregunta pide uno",
    "title": "Cifra"
  },
  "unidad": {
    "anyOf": [
      {
        "type": "string"
      },
      {
        "type": "null"
      }
    ],
    "default": null,
    "description": "USD, shares, porcentaje…",
    "title": "Unidad"
  },
  "ticker": {
    "anyOf": [
      ...


In [ ]:
# El mismo agente, en cinco líneas.
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langgraph.checkpoint.memory import InMemorySaver

agente = None
if HAY_CLAVE:
    agente = create_agent(
        model=modelo,
        tools=HERRAMIENTAS,
        system_prompt=SYSTEM,
        response_format=ToolStrategy(schema=RespuestaFinanciera),
        checkpointer=InMemorySaver(),
    )
    print("Agente montado con", len(HERRAMIENTAS), "herramientas.")
else:
    print("Sin clave: no se puede montar el agente.")

# Qué hace cada línea que vosotros hicisteis a mano:
#
#   tools=            los bind_tools y el diccionario POR_NOMBRE
#   response_format=  la validación de la salida, que no teníais
#   checkpointer=     la memoria entre invocaciones, que tampoco teníais
#   (el bucle)        las treinta líneas de la celda 23
#
# Y trae cosas que no habíais escrito: reintentos, streaming, callbacks para
# instrumentar la ejecución, e interrupciones para humano en el medio, que es
# de lo que va el día 17.
#
# Si el modelo que elijáis no soporta salida estructurada nativa, esa línea
# falla. La salida es envolverlo:
#     from langchain.agents.structured_output import ToolStrategy
#     response_format=ToolStrategy(schema=RespuestaFinanciera)

Agente montado con 4 herramientas.


In [ ]:
# Memoria: dos invocaciones con el mismo thread_id.
#
# `thread_id` es lo que convierte dos llamadas sueltas en una conversación.
# Sin checkpointer no hay memoria; sin thread_id, el checkpointer no sabe
# a qué conversación pertenece cada invocación.
CONFIG = {"configurable": {"thread_id": "clase-s1"}}

if agente is not None:
    r1 = agente.invoke(
        {"messages": [{"role": "user", "content":
                       "¿Cuál fue el revenue de NVIDIA en FY2025?"}]},
        config=CONFIG,
    )
    print("1 >", r1["structured_response"].respuesta)

    # Esta segunda pregunta no nombra ni la compañía ni el ejercicio.
    r2 = agente.invoke(
        {"messages": [{"role": "user",
                       "content": "¿Y cuánto es eso comparado con el "
                                  "ejercicio anterior?"}]},
        config=CONFIG,
    )
    print("2 >", r2["structured_response"].respuesta)
    print(f"\nmensajes acumulados en el hilo: {len(r2['messages'])}")
else:
    r1 = r2 = None
    print("Sin clave: esta celda necesita el agente.")

# Cambiad el thread_id de la segunda invocación y volved a ejecutar: la
# pregunta de seguimiento deja de tener sentido. Eso es exactamente lo que
# le pasa a un agente sin memoria.

1 > El revenue (ingresos totales) de NVIDIA en el ejercicio fiscal 2025 (FY2025) fue de 130.497.000.000 USD (cierre a 26 de enero de 2025).
2 > En el ejercicio anterior (FY2024), el revenue fue de 60.922.000.000 USD. Frente a los 130.497.000.000 USD de FY2025, supone un aumento de 69.575.000.000 USD, lo que representa un crecimiento del 114,20%.

mensajes acumulados en el hilo: 10


## NOTA (histórico): el fallback no cubría create_agent con salida estructurada

Mientras usábamos la cascada, detectamos que `with_fallbacks` funcionaba
bien en `agente_manual` y en las llamadas normales de herramientas dentro
de `create_agent`, pero **no** en la generación de la respuesta
estructurada final — probablemente un paso interno de LangGraph que
reinvoca el modelo base directamente. Probado con `response_format`
nativo y con `ToolStrategy`, mismo resultado.

Con el pago activado y la cascada retirada, este problema deja de
aplicar: `create_agent` usa directamente `modelo` (Gemini de pago), sin
ningún fallback de por medio.

In [ ]:
# Ver la trayectoria.
#
# Una respuesta no se puede juzgar sin ver el camino. Esta función imprime la
# trayectoria: qué herramientas se llamaron, en qué orden, con qué argumentos
# y qué devolvió cada una.
#
# Es también la pieza que hace posible el evaluador `uso_la_tool_correcta` del
# día 17: sin trayectoria no se puede distinguir una respuesta correcta de una
# respuesta correcta por casualidad.
def pretty_trace(resultado) -> None:
    """Imprime la trayectoria: qué herramientas se llamaron, con qué
    argumentos y qué devolvieron."""
    pendientes = {}
    n = 0
    for mensaje in resultado["messages"]:
        for tc in getattr(mensaje, "tool_calls", None) or []:
            n += 1
            pendientes[tc["id"]] = n
            print(f"{n}. {tc['name']}({tc['args']})")
        id_llamada = getattr(mensaje, "tool_call_id", None)
        if id_llamada in pendientes:
            texto = str(mensaje.content).replace("\n", " ")
            print(f"   -> {texto[:160]}"
                  f"{'...' if len(texto) > 160 else ''}")
    print(f"\n{n} llamadas a herramienta")
    e = resultado.get("structured_response")
    if e is not None:
        print(f"respuesta: {e.respuesta}")
        print(f"fuente: {e.fuente} · cifra: {e.cifra} {e.unidad or ''} "
              f"· cita: {e.chunk_id}")


if r2 is not None:
    pretty_trace(r2)

1. get_xbrl_fact({'ticker': 'NVDA', 'fiscal_year': 2025, 'concept': 'Revenues'})
   -> NVDA FY2025 · Revenues = 130,497,000,000 USD (cierre de ejercicio 2025-01-26, según el 10-K)
2. RespuestaFinanciera({'respuesta': 'El revenue (ingresos totales) de NVIDIA en el ejercicio fiscal 2025 (FY2025) fue de 130.497.000.000 USD (cierre a 26 de enero de 2025).', 'fuente': 'xbrl', 'unidad': 'USD', 'ticker': 'NVDA', 'cifra': 130497000000, 'ejercicio': 2025})
   -> Returning structured response: respuesta='El revenue (ingresos totales) de NVIDIA en el ejercicio fiscal 2025 (FY2025) fue de 130.497.000.000 USD (cierre a 26 d...
3. get_xbrl_fact({'fiscal_year': 2024, 'ticker': 'NVDA', 'concept': 'Revenues'})
   -> NVDA FY2024 · Revenues = 60,922,000,000 USD (cierre de ejercicio 2024-01-28, según el 10-K)
4. RespuestaFinanciera({'unidad': 'USD', 'fuente': 'xbrl', 'cifra': 69575000000, 'ticker': 'NVDA', 'ejercicio': 2025, 'respuesta': 'En el ejercicio anterior (FY2024), el revenue fue de 60.922.000.000

In [ ]:
# --- verificación de §6 --------------------------------------------------
if agente is None:
    print("Sin clave: §6 no se puede verificar. El resto del notebook sí.")
else:
    prueba = agente.invoke(
        {"messages": [{"role": "user", "content":
                       "¿Cuál fue el revenue de NVIDIA en FY2024?"}]},
        config={"configurable": {"thread_id": "verificacion-s1"}},
    )
    e = prueba["structured_response"]

    assert isinstance(e, RespuestaFinanciera), \
        "structured_response debería ser una RespuestaFinanciera validada."
    assert e.fuente in {"xbrl", "texto", "ambas", "ninguna"}
    assert any(
        tc["name"] == "get_xbrl_fact"
        for m in prueba["messages"]
        for tc in (getattr(m, "tool_calls", None) or [])
    ), ("Una pregunta numérica que no pasó por get_xbrl_fact es un fallo, "
        "aunque el número sea correcto. Revisad el system prompt.")

    print("§6 listo:", e.respuesta)
    print(f"cifra={e.cifra} unidad={e.unidad} fuente={e.fuente}")

§6 listo: El revenue total de NVIDIA en el ejercicio fiscal 2024 (FY2024, finalizado el 28 de enero de 2024) fue de 60.922.000.000 USD.
cifra=60922000000.0 unidad=USD fuente=xbrl


## La práctica

El enunciado completo lo tenéis en la mano. Aquí queda lo imprescindible.

**Qué se entrega**, en grupos de 3:

| | Peso |
| --- | --- |
| Repositorio con el agente, el golden set y los evaluadores | 30 |
| Presentación de 8 minutos el 24, con 10 preguntas ciegas en directo | 70 |

**El golden set.** Recibís 20 preguntas oficiales y escribís **20 vuestras**,
de las cuales **al menos 6 comparativas**. Tres familias:

| Familia | Qué mide | Campos que se rellenan |
| --- | --- | --- |
| `extractiva` | Retrieval y trazabilidad | `item_esperado`, `ancla_texto` |
| `numerica` | El guardrail contra XBRL | `cifra_esperada`, `unidad`, `concept_xbrl` |
| `comparativa` | Que el agente descomponga y compare | los tres, por cada ejercicio |

**Por qué el mínimo de 6 comparativas.** «¿Qué riesgos nuevos añadió entre
FY2024 y FY2025?» no la contesta una sola recuperación: hay que descomponer,
recuperar dos veces y comparar. Es la familia donde el agente deja de ser
decoración sobre una *pipeline*, y sin ella vuestro informe no puede demostrar
que hiciera falta un agente. Hay sustancia real que encontrar: el Item 1A
cambia entre el 49 % y el 85 % de los párrafos entre los dos ejercicios, según
la compañía.

**La verdad se ancla a una frase, no a un `chunk_id`.** El día 17 vais a
cambiar el troceado, y en cuanto lo toquéis todos los `chunk_id` son otros. Si
la métrica dependiera de ellos, el grupo que mejorase el troceado saldría
penalizado por haberlo mejorado. Por eso `ancla_texto` es un texto literal del
informe: **una frase**, no tres párrafos.

In [ ]:
import shutil, pathlib

origen_drive = pathlib.Path("/content/drive/MyDrive/MIAX_2026/golden_set_ejemplo.jsonl")
if origen_drive.exists():
    shutil.copy(origen_drive, "/content/golden_set_ejemplo.jsonl")
    print("Copiado desde Drive.")
else:
    print("No está en Drive tampoco — hay que subirlo manualmente.")

Copiado desde Drive.


In [ ]:
# El golden set oficial: las 20 preguntas con respuesta conocida.
from pathlib import Path

RUTA_GOLDEN = Path("golden_set.jsonl")
if not RUTA_GOLDEN.is_file():
    RUTA_GOLDEN = Path("golden_set_ejemplo.jsonl")   # provisional

golden = [json.loads(l) for l in open(RUTA_GOLDEN, encoding="utf-8")
          if l.strip()]
g = pd.DataFrame(golden)

print(f"{RUTA_GOLDEN.name}: {len(g)} preguntas")
print("\nreparto por familia:")
print(g.familia.value_counts().to_string())
print("\nherramienta esperada:")
print(g.herramienta_esperada.explode().value_counts().to_string())

for fila in golden[:2]:
    print("\n" + json.dumps(fila, ensure_ascii=False, indent=2))

# NOTA PARA EL 10 DE SEPTIEMBRE: si arriba pone `golden_set_ejemplo.jsonl`,
# es que las 20 oficiales todavía no están en la carpeta. El fichero de
# ejemplo tiene tres preguntas y sirve solo para ver el esquema y probar el
# validador de la celda siguiente.

golden_set_ejemplo.jsonl: 3 preguntas

reparto por familia:
familia
numerica       1
extractiva     1
comparativa    1

herramienta esperada:
herramienta_esperada
get_xbrl_fact     2
search_filings    1

{
  "id": "ej-001",
  "pregunta": "¿Cuál fue el revenue de NVIDIA en el ejercicio 2024?",
  "familia": "numerica",
  "ticker": "NVDA",
  "fiscal_year": 2024,
  "respuesta_esperada": "60.922 millones de dólares",
  "cifra_esperada": 60922000000.0,
  "unidad": "USD",
  "concept_xbrl": "Revenues",
  "item_esperado": null,
  "ancla_texto": null,
  "ancla_inicio": null,
  "ancla_fin": null,
  "chunk_id_esperado": null,
  "herramienta_esperada": [
    "get_xbrl_fact"
  ],
  "autor": "ejemplo"
}

{
  "id": "ej-002",
  "pregunta": "¿Qué dice Microsoft en FY2025 sobre el uso indebido de sus sistemas de IA?",
  "familia": "extractiva",
  "ticker": "MSFT",
  "fiscal_year": 2025,
  "respuesta_esperada": "Que sus sistemas de IA ofrecen capacidades potentes que pueden usarse indebidamente, con conse

In [ ]:
# Vuestras 20 preguntas, y el validador que tienen que pasar antes de
# entregarlas. Un golden set que no pasa el validador no se corrige: se
# devuelve.
PLANTILLA = {
    "id": "g3-001",
    "pregunta": "¿Cuál fue el revenue de NVIDIA en el ejercicio 2024?",
    "familia": "numerica",              # extractiva | numerica | comparativa
    "ticker": "NVDA",
    "fiscal_year": 2024,
    "respuesta_esperada": "60.922 millones de dólares",
    "cifra_esperada": 60922000000.0,
    "unidad": "USD",
    "concept_xbrl": "Revenues",
    "item_esperado": None,
    "ancla_texto": None,
    "ancla_inicio": None,
    "ancla_fin": None,
    "chunk_id_esperado": None,
    "herramienta_esperada": ["get_xbrl_fact"],
    "autor": "grupo-3",
}

CAMPOS = set(PLANTILLA)
FAMILIAS = {"extractiva", "numerica", "comparativa"}


def validar(preguntas: list[dict], exigir_20: bool = True) -> list[str]:
    """Los problemas del fichero, uno por línea. Lista vacía = correcto."""
    problemas = []
    tickers = set(secciones.ticker)
    ejercicios = set(secciones.fiscal_year.astype(int))
    vistos = set()

    for p in preguntas:
        pid = p.get("id", "(sin id)")
        if faltan := CAMPOS - set(p):
            problemas.append(f"{pid}: faltan campos {sorted(faltan)}")
            continue
        if p["id"] in vistos:
            problemas.append(f"{pid}: id repetido")
        vistos.add(p["id"])
        if p["familia"] not in FAMILIAS:
            problemas.append(f"{pid}: familia '{p['familia']}' no válida")
        if p["ticker"] not in tickers:
            problemas.append(f"{pid}: {p['ticker']} no está en el corpus")
        if int(p["fiscal_year"]) not in ejercicios:
            problemas.append(f"{pid}: FY{p['fiscal_year']} no está en el "
                             f"corpus")
        if p["familia"] in {"numerica", "comparativa"}:
            if p.get("cifra_esperada") is None:
                problemas.append(f"{pid}: numérica sin cifra_esperada")
            concepto = p.get("concept_xbrl")
            hay = xbrl[(xbrl.ticker == p["ticker"])
                       & (xbrl.fiscal_year == int(p["fiscal_year"]))
                       & (xbrl.concept == concepto)]
            if concepto and hay.empty:
                problemas.append(
                    f"{pid}: {p['ticker']} no reporta '{concepto}' en "
                    f"FY{p['fiscal_year']}. El concepto se mira en "
                    f"xbrl_facts.parquet, nunca por analogía con otra "
                    f"compañía.")
        if p["familia"] in {"extractiva", "comparativa"}:
            ancla = p.get("ancla_texto")
            if not ancla:
                problemas.append(f"{pid}: extractiva sin ancla_texto")
            elif len(ancla.split()) > 40:
                problemas.append(
                    f"{pid}: ancla de {len(ancla.split())} palabras. Una "
                    f"frase. Así no medís vuestro retrieval, medís vuestro "
                    f"tamaño de ventana.")
        if not p.get("herramienta_esperada"):
            problemas.append(f"{pid}: sin herramienta_esperada")

    if exigir_20:
        if len(preguntas) != 20:
            problemas.append(f"hacen falta 20 preguntas, hay {len(preguntas)}")
        n_comp = sum(p.get("familia") == "comparativa" for p in preguntas)
        if n_comp < 6:
            problemas.append(f"hacen falta 6 comparativas, hay {n_comp}")
    return problemas


problemas = validar(golden, exigir_20=False)
print("Validando el fichero cargado:")
print("\n".join(f"  - {p}" for p in problemas) or "  sin problemas")

# --- verificación de §7 --------------------------------------------------
assert not validar([PLANTILLA], exigir_20=False), \
    "La plantilla debería pasar su propio validador."
assert validar([{**PLANTILLA, "ticker": "TSLA"}], exigir_20=False), \
    "El validador tiene que rechazar una compañía que no está en el corpus."
print("\n§7 listo. El validador funciona; ahora escribid las preguntas.")

Validando el fichero cargado:
  sin problemas

§7 listo. El validador funciona; ahora escribid las preguntas.


## Para el jueves 17

Dos cosas, y sin ellas la sesión que viene no os rinde:

1. **El baseline corriendo.** Este notebook, ejecutado de arriba abajo en
   vuestro repositorio, con `responder()` y `evaluar()` como dice el §6 del
   enunciado. El día 17 empieza ejecutando vuestro agente contra cinco
   preguntas duras y viendo en qué falla: si llegáis sin agente, la primera
   hora se convierte en soporte técnico y la perdéis.
2. **Vuestras 20 preguntas escritas**, pasando el validador de la celda
   anterior, con al menos 6 comparativas.

> Si el baseline no os arranca el día 17, hay un notebook de rescate que lo
> reconstruye en tres minutos. Existe para que nadie se quede fuera, no para
> ahorrarse el trabajo: quien lo use empieza la sesión sin conocer su propio
> código.

## Qué se lleva de hoy

- Una herramienta no es una función: es una **función más un docstring**, y el
  docstring es lo que decide si se llama. La descripción es *prompt
  engineering*, no documentación.
- Un agente es un **bucle `while`** alrededor de una llamada a un modelo. Lo
  habéis escrito. `create_agent` hace lo mismo con reintentos, memoria,
  validación y trazas.
- El retrieval no es la arquitectura: es **una herramienta más**, y compite
  con una consulta a XBRL que es mil veces más barata y exacta. Enrutar bien
  entre las dos es el trabajo.
- Un agente sin límites entra en bucle. Lo habéis visto. El día 17 se arregla.

## Qué falta, y cuándo llega

| Falta | Cuándo |
| --- | --- |
| Qué hay dentro de `search_filings`, y cómo medirlo | 17 sep |
| Que el agente no se invente cifras | 17 sep |
| Cortar el bucle infinito | 17 sep |
| Evaluar de verdad, con el golden set | 17 sep |
| ReAct, el paper y el nombre de lo que habéis escrito | 18 sep |
| MCP, ADK y A2A | 19 sep, más un notebook de bonus |

## Golden Set

In [ ]:
import re, json

NUMERICAS = [
    ("N1", "¿Cuál fue el beneficio neto de Amazon en FY2025?", "AMZN", 2025, "NetIncomeLoss"),
    ("N2", "¿Cuáles fueron los ingresos de NVIDIA (ejercicio que cerró en enero de 2025)?", "NVDA", 2025, "Revenues"),
    ("N3", "¿Cuál fue el beneficio neto de Microsoft en FY2024?", "MSFT", 2024, "NetIncomeLoss"),
    ("N4", "¿Cuál fue el total de activos de Apple en FY2025?", "AAPL", 2025, "Assets"),
    ("N5", "¿Cuál fue el resultado operativo de Alphabet en FY2024?", "GOOGL", 2024, "OperatingIncomeLoss"),
    ("N6", "¿Cuáles fueron los ingresos totales de Meta en FY2025?", "META", 2025, "RevenueFromContractWithCustomerExcludingAssessedTax"),
    ("N7", "¿Cuál fue el beneficio neto de NVIDIA en FY2025?", "NVDA", 2025, "NetIncomeLoss"),
    ("N8", "¿Cuáles fueron los ingresos totales de Amazon en FY2024?", "AMZN", 2024, "RevenueFromContractWithCustomerExcludingAssessedTax"),
]

print("=" * 70)
print("NUMÉRICAS — resultado real de get_xbrl_fact")
print("=" * 70)
for id_, pregunta, ticker, fy, concept in NUMERICAS:
    salida = get_xbrl_fact.invoke({"ticker": ticker, "fiscal_year": fy, "concept": concept})
    print(f"\n[{id_}] {ticker} FY{fy} · {concept}\n  → {salida}")

# --- Preguntas EXTRACTIVAS: (id, pregunta, ticker, fiscal_year, item, query) ---
EXTRACTIVAS = [
    ("E1", "¿Qué riesgos menciona Apple sobre su cadena de suministro?", "AAPL", 2025, "1A", "supply chain risk manufacturing"),
    ("E2", "¿Qué riesgos regulatorios/antitrust menciona Alphabet?", "GOOGL", 2025, "1A", "antitrust regulatory competition risk"),
    ("E3", "¿Qué riesgos de privacidad/datos menciona Meta?", "META", 2025, "1A", "privacy data regulation risk"),
    ("E4", "¿Qué riesgos de mercado menciona NVIDIA?", "NVDA", 2025, "7A", "market risk foreign currency interest rate"),
    ("E5", "¿Qué dice Microsoft sobre asignación de capital/liquidez?", "MSFT", 2025, "7", "capital allocation liquidity strategy"),
    ("E6", "¿Qué riesgos de plantilla/logística menciona Amazon?", "AMZN", 2025, "1A", "workforce fulfillment network risk"),
]

print("\n" + "=" * 70)
print("EXTRACTIVAS — top fragmentos de search_filings (k=3)")
print("=" * 70)
resultados_extractivas = {}
for id_, pregunta, ticker, fy, item, query in EXTRACTIVAS:
    salida = search_filings.invoke({"query": query, "ticker": ticker, "fiscal_year": fy, "item": item, "k": 3})
    resultados_extractivas[id_] = salida
    print(f"\n[{id_}] {ticker} FY{fy} · item {item} · query: '{query}'")
    print(f"  → {salida}")

print("\n\nListo. Copiad manualmente de aquí arriba:")
print("- Numéricas: la cifra exacta (sin comas) → cifra_esperada")
print("- Extractivas: el chunk_id del fragmento más relevante → chunk_id_esperado")
print("  y una frase corta literal de su texto → ancla_texto")

NUMÉRICAS — resultado real de get_xbrl_fact

[N1] AMZN FY2025 · NetIncomeLoss
  → AMZN FY2025 · NetIncomeLoss = 77,670,000,000 USD (cierre de ejercicio 2025-12-31, según el 10-K)

[N2] NVDA FY2025 · Revenues
  → NVDA FY2025 · Revenues = 130,497,000,000 USD (cierre de ejercicio 2025-01-26, según el 10-K)

[N3] MSFT FY2024 · NetIncomeLoss
  → MSFT FY2024 · NetIncomeLoss = 88,136,000,000 USD (cierre de ejercicio 2024-06-30, según el 10-K)

[N4] AAPL FY2025 · Assets
  → AAPL FY2025 · Assets = 359,241,000,000 USD (cierre de ejercicio 2025-09-27, según el 10-K)

[N5] GOOGL FY2024 · OperatingIncomeLoss
  → GOOGL FY2024 · OperatingIncomeLoss = 112,390,000,000 USD (cierre de ejercicio 2024-12-31, según el 10-K)

[N6] META FY2025 · RevenueFromContractWithCustomerExcludingAssessedTax
  → META FY2025 · RevenueFromContractWithCustomerExcludingAssessedTax = 200,966,000,000 USD (cierre de ejercicio 2025-12-31, según el 10-K)

[N7] NVDA FY2025 · NetIncomeLoss
  → NVDA FY2025 · NetIncomeLoss = 72,880,0

In [ ]:
def calcular_ancla(ticker, fiscal_year, item, frase_ancla):
    texto_seccion = read_section.invoke({"ticker": ticker, "fiscal_year": fiscal_year, "item": item})
    inicio = texto_seccion.find(frase_ancla)
    if inicio == -1:
        print(f"⚠️  NO ENCONTRADA en {ticker} FY{fiscal_year} item {item}: '{frase_ancla[:50]}...'")
        return None, None
    return inicio, inicio + len(frase_ancla)

anclas = [
    {"id": "E1", "pregunta": "¿Qué riesgos menciona Apple sobre su cadena de suministro?", "familia": "extractiva", "ticker": "AAPL", "fiscal_year": 2025, "respuesta_esperada": "Apple depende de fabricantes y proveedores únicos, lo que reduce su control directo sobre producción y distribución y la expone a riesgos de suministro y precios.", "cifra_esperada": None, "unidad": None, "concept_xbrl": None, "item_esperado": "1A", "ancla_texto": "The Company\u2019s operations are also subject to the risks of industrial accidents at its suppliers and contract manufacturers.", "ancla_inicio": 8642, "ancla_fin": 8765, "chunk_id_esperado": "AAPL-2025-1A-0004", "herramienta_esperada": ["search_filings"], "autor": "andrea"},
    {"id": "E2", "pregunta": "¿Qué riesgos regulatorios o de competencia (antitrust) menciona Alphabet?", "familia": "extractiva", "ticker": "GOOGL", "fiscal_year": 2025, "respuesta_esperada": "Alphabet enfrenta demandas antitrust del DOJ y varios estados sobre sus prácticas de búsqueda y publicidad, con sentencias que imponen restricciones a su distribución de servicios.", "cifra_esperada": None, "unidad": None, "concept_xbrl": None, "item_esperado": "1A", "ancla_texto": "the DOJ and a number of state Attorneys General filed a lawsuit concerning our Search and Search advertising practices and our compliance with US antitrust laws.", "ancla_inicio": 58099, "ancla_fin": 58260, "chunk_id_esperado": "GOOGL-2025-1A-0026", "herramienta_esperada": ["search_filings"], "autor": "andrea"},
    {"id": "E3", "pregunta": "¿Qué riesgos de privacidad o regulación de datos menciona Meta?", "familia": "extractiva", "ticker": "META", "fiscal_year": 2025, "respuesta_esperada": "Meta enfrenta regulaciones evolutivas de privacidad (GDPR, DMA, DSA, CCPA) que limitan su uso de datos y publicidad, con riesgo de sanciones significativas.", "cifra_esperada": None, "unidad": None, "concept_xbrl": None, "item_esperado": "1A", "ancla_texto": "complex and evolving U.S. and foreign privacy, data use, data combination, data protection, content and content moderation, competition, youth, safety, consumer protection, advertising, and other laws and regulations", "ancla_inicio": 2598, "ancla_fin": 2814, "chunk_id_esperado": "META-2025-1A-0001", "herramienta_esperada": ["search_filings"], "autor": "andrea"},
    {"id": "E4", "pregunta": "¿Qué riesgos de tipo de cambio menciona NVIDIA?", "familia": "extractiva", "ticker": "NVDA", "fiscal_year": 2025, "respuesta_esperada": "NVIDIA considera mínima su exposición directa al riesgo cambiario porque casi todas sus ventas están en dólares y usa contratos forward para cubrirse.", "cifra_esperada": None, "unidad": None, "concept_xbrl": None, "item_esperado": "7A", "ancla_texto": "We consider our direct exposure to foreign exchange rate fluctuations to be minimal as substantially all of our sales are in United States dollars", "ancla_inicio": 1113, "ancla_fin": 1259, "chunk_id_esperado": "NVDA-2025-7A-0001", "herramienta_esperada": ["search_filings"], "autor": "andrea"},
    {"id": "E5", "pregunta": "¿Qué dice Microsoft sobre sus planes de uso de capital?", "familia": "extractiva", "ticker": "MSFT", "fiscal_year": 2025, "respuesta_esperada": "Microsoft seguirá invirtiendo en ventas, marketing, infraestructura y centros de datos, incluidas inversiones en infraestructura y entrenamiento de IA.", "cifra_esperada": None, "unidad": None, "concept_xbrl": None, "item_esperado": "7", "ancla_texto": "We will continue to invest in capital expenditures to support growth in our cloud offerings and our investments in AI infrastructure and training.", "ancla_inicio": 35973, "ancla_fin": 36119, "chunk_id_esperado": "MSFT-2025-7-0019", "herramienta_esperada": ["search_filings"], "autor": "andrea"},
    {"id": "E6", "pregunta": "¿Qué riesgos relacionados con su red de cumplimiento (fulfillment) menciona Amazon?", "familia": "extractiva", "ticker": "AMZN", "fiscal_year": 2025, "respuesta_esperada": "Amazon enfrenta riesgos si no predice correctamente la demanda, lo que puede causar capacidad insuficiente o excesiva en su red de fulfillment y centros de datos.", "cifra_esperada": None, "unidad": None, "concept_xbrl": None, "item_esperado": "1A", "ancla_texto": "Failures to adequately predict customer demand and consumer spending patterns or otherwise optimize and operate our fulfillment network and data centers successfully", "ancla_inicio": 25192, "ancla_fin": 25357, "chunk_id_esperado": "AMZN-2025-1A-0012", "herramienta_esperada": ["search_filings"], "autor": "andrea"},
]

for pregunta in anclas:
    i, f = calcular_ancla(pregunta["ticker"], pregunta["fiscal_year"],
                           pregunta["item_esperado"], pregunta["ancla_texto"])
    coincide = (i == pregunta["ancla_inicio"] and f == pregunta["ancla_fin"])
    print(f"{pregunta['id']}: calculado=({i},{f}) guardado=({pregunta['ancla_inicio']},{pregunta['ancla_fin']}) {'✅' if coincide else '❌'}")

E1: calculado=(8642,8765) guardado=(8642,8765) ✅
E2: calculado=(58099,58260) guardado=(58099,58260) ✅
E3: calculado=(2598,2814) guardado=(2598,2814) ✅
E4: calculado=(1113,1259) guardado=(1113,1259) ✅
E5: calculado=(35973,36119) guardado=(35973,36119) ✅
E6: calculado=(25192,25357) guardado=(25192,25357) ✅


---

## REVIEW: Criterio detrás de nuestro golden set

Estas preguntas no salieron al azar. Las elegimos para que el golden set
haga algo más que "hacer 20 preguntas": queríamos que pusiera a prueba las
trampas que el propio corpus documenta, y que cubriera las 6 empresas y
los 4 items de forma equilibrada.

**Por qué incluimos preguntas que sabíamos que iban a "fallar".**
Metimos a propósito Amazon + `GrossProfit` (N1): sabíamos que Amazon no
reporta ese concepto. No es un error nuestro, es la pregunta — queremos
que el golden set premie a un agente que dice "no está en el corpus" en
vez de inventar o derivar una cifra. Es la misma idea que el enunciado
señala como fallo típico: acertar el número por el camino equivocado
cuenta como fallo.

**Por qué Revenues nos hizo cambiar de concepto en dos preguntas.**
Al intentar sacar los ingresos de Meta (N6) y Amazon (N8) con
`concept="Revenues"`, la herramienta nos dijo que ese concepto no existe
para esas empresas — el nombre real es
`RevenueFromContractWithCustomerExcludingAssessedTax`. Lo dejamos así a
propósito en el golden set: es justo el tipo de variación de vocabulario
XBRL entre empresas que el agente tiene que resolver solo, reintentando
si hace falta.

**Por qué el ancla es una frase, no un chunk_id.** El propio enunciado
(§ La práctica) lo explica: el día 17 vamos a tocar el troceado del
retrieval, y en cuanto lo hagamos, todos los `chunk_id` cambian. Si
ancláramos la verdad a un `chunk_id`, mejorar el troceado nos penalizaría
en vez de premiarnos. Por eso verificamos cada `ancla_texto` contra el
texto real de la sección (con `ancla_inicio`/`ancla_fin`), no contra el
identificador del fragmento.

**Un hallazgo que no esperábamos.** Al verificar el ancla de E1 (Apple),
la búsqueda literal falló la primera vez porque el documento usa una
comilla tipográfica curva (`’`) en vez de un apóstrofo recto (`'`). Nos
sirvió de aviso: una cita "casi idéntica" no es una cita verificable — el
evaluador de citas del §7 exige coincidencia exacta de carácter.

In [ ]:
COMPARATIVAS = [
    ("C1", "NVDA", "Revenues", "7", "revenue growth data center AI demand"),
    ("C2", "MSFT", "RevenueFromContractWithCustomerExcludingAssessedTax", "1A", "agentic AI systems autonomous actions human review"),
    ("C3", "AAPL", "RevenueFromContractWithCustomerExcludingAssessedTax", "7", "net sales growth Services Products"),
    ("C4", "GOOGL", "OperatingIncomeLoss", "7", "operating income margin growth"),
    ("C5", "META", "ResearchAndDevelopmentExpense", "7", "research and development expense increase AI infrastructure"),
    ("C6", "AMZN", "NetIncomeLoss", "7", "net income growth operating efficiency"),
]

print("=" * 70)
print("COMPARATIVAS — cifras XBRL (FY2024 vs FY2025)")
print("=" * 70)
for id_, ticker, concept, item, query in COMPARATIVAS:
    print(f"\n=== {id_} · {ticker} · {concept} ===")
    for fy in [2024, 2025]:
        salida = get_xbrl_fact.invoke({"ticker": ticker, "fiscal_year": fy, "concept": concept})
        print(f"  FY{fy}: {salida}")

print("\n" + "=" * 70)
print("COMPARATIVAS — candidatos de ancla_texto (search_filings, FY2025, k=3)")
print("=" * 70)
for id_, ticker, concept, item, query in COMPARATIVAS:
    print(f"\n=== {id_} · {ticker} FY2025 item {item} · query: '{query}' ===")
    salida = search_filings.invoke({"query": query, "ticker": ticker, "fiscal_year": 2025, "item": item, "k": 3})
    print(salida)

COMPARATIVAS — cifras XBRL (FY2024 vs FY2025)

=== C1 · NVDA · Revenues ===
  FY2024: NVDA FY2024 · Revenues = 60,922,000,000 USD (cierre de ejercicio 2024-01-28, según el 10-K)
  FY2025: NVDA FY2025 · Revenues = 130,497,000,000 USD (cierre de ejercicio 2025-01-26, según el 10-K)

=== C2 · MSFT · RevenueFromContractWithCustomerExcludingAssessedTax ===
  FY2024: MSFT FY2024 · RevenueFromContractWithCustomerExcludingAssessedTax = 245,122,000,000 USD (cierre de ejercicio 2024-06-30, según el 10-K)
  FY2025: MSFT FY2025 · RevenueFromContractWithCustomerExcludingAssessedTax = 281,724,000,000 USD (cierre de ejercicio 2025-06-30, según el 10-K)

=== C3 · AAPL · RevenueFromContractWithCustomerExcludingAssessedTax ===
  FY2024: AAPL FY2024 · RevenueFromContractWithCustomerExcludingAssessedTax = 391,035,000,000 USD (cierre de ejercicio 2024-09-28, según el 10-K)
  FY2025: AAPL FY2025 · RevenueFromContractWithCustomerExcludingAssessedTax = 416,161,000,000 USD (cierre de ejercicio 2025-09-27, segú

In [ ]:
anclas_comparativas = [
    {"id": "C1", "ticker": "NVDA", "fiscal_year": 2025, "item": "7",
     "texto": "Revenue growth in fiscal year 2025 was driven by data center compute and networking platforms for accelerated computing and AI solutions."},
    {"id": "C2", "ticker": "MSFT", "fiscal_year": 2025, "item": "1A",
     "texto": "Human review of certain inputs and outputs may be required, including for agentic AI systems that can take actions autonomously."},
    {"id": "C3", "ticker": "AAPL", "fiscal_year": 2025, "item": "7",
     "texto": "Services net sales increased during 2025 compared to 2024 primarily due to higher net sales from advertising, the App Store and cloud services."},
    {"id": "C4", "ticker": "GOOGL", "fiscal_year": 2025, "item": "7",
     "texto": "Google Services operating income increased $18.1 billion from 2024 to 2025."},
    {"id": "C5", "ticker": "META", "fiscal_year": 2025, "item": "7",
     "texto": "Research and development expenses in 2025 increased $13.50 billion, or 31%, compared to 2024."},
    {"id": "C6", "ticker": "AMZN", "fiscal_year": 2025, "item": "7",
     "texto": "Operating income was $68.6 billion and $80.0 billion for 2024 and 2025."},
]

for a in anclas_comparativas:
    i, f = calcular_ancla(a["ticker"], a["fiscal_year"], a["item"], a["texto"])
    n_palabras = len(a["texto"].split())
    print(f"{a['id']}: inicio={i}, fin={f}, palabras={n_palabras}")

C1: inicio=1444, fin=1581, palabras=21
C2: inicio=35169, fin=35297, palabras=20
C3: inicio=6437, fin=6580, palabras=23
C4: inicio=29467, fin=29542, palabras=11
C5: inicio=41958, fin=42051, palabras=14
C6: inicio=35756, fin=35827, palabras=12


In [ ]:
golden_set_completo = [
    # --- Numéricas (8) ---
    {"id": "N1", "pregunta": "¿Cuál fue el beneficio neto de Amazon en FY2025?", "familia": "numerica", "ticker": "AMZN", "fiscal_year": 2025, "respuesta_esperada": "77.670 millones de dólares", "cifra_esperada": 77670000000.0, "unidad": "USD", "concept_xbrl": "NetIncomeLoss", "item_esperado": None, "ancla_texto": None, "ancla_inicio": None, "ancla_fin": None, "chunk_id_esperado": None, "herramienta_esperada": ["get_xbrl_fact"], "autor": "andrea"},
    {"id": "N2", "pregunta": "¿Cuáles fueron los ingresos de NVIDIA correspondientes al ejercicio fiscal que cerró en enero de 2025?", "familia": "numerica", "ticker": "NVDA", "fiscal_year": 2025, "respuesta_esperada": "130.497 millones de dólares", "cifra_esperada": 130497000000.0, "unidad": "USD", "concept_xbrl": "Revenues", "item_esperado": None, "ancla_texto": None, "ancla_inicio": None, "ancla_fin": None, "chunk_id_esperado": None, "herramienta_esperada": ["get_xbrl_fact"], "autor": "andrea"},
    {"id": "N3", "pregunta": "¿Cuál fue el beneficio neto de Microsoft en FY2024?", "familia": "numerica", "ticker": "MSFT", "fiscal_year": 2024, "respuesta_esperada": "88.136 millones de dólares", "cifra_esperada": 88136000000.0, "unidad": "USD", "concept_xbrl": "NetIncomeLoss", "item_esperado": None, "ancla_texto": None, "ancla_inicio": None, "ancla_fin": None, "chunk_id_esperado": None, "herramienta_esperada": ["get_xbrl_fact"], "autor": "andrea"},
    {"id": "N4", "pregunta": "¿Cuál fue el total de activos de Apple en FY2025?", "familia": "numerica", "ticker": "AAPL", "fiscal_year": 2025, "respuesta_esperada": "359.241 millones de dólares", "cifra_esperada": 359241000000.0, "unidad": "USD", "concept_xbrl": "Assets", "item_esperado": None, "ancla_texto": None, "ancla_inicio": None, "ancla_fin": None, "chunk_id_esperado": None, "herramienta_esperada": ["get_xbrl_fact"], "autor": "andrea"},
    {"id": "N5", "pregunta": "¿Cuál fue el resultado operativo de Alphabet en FY2024?", "familia": "numerica", "ticker": "GOOGL", "fiscal_year": 2024, "respuesta_esperada": "112.390 millones de dólares", "cifra_esperada": 112390000000.0, "unidad": "USD", "concept_xbrl": "OperatingIncomeLoss", "item_esperado": None, "ancla_texto": None, "ancla_inicio": None, "ancla_fin": None, "chunk_id_esperado": None, "herramienta_esperada": ["get_xbrl_fact"], "autor": "andrea"},
    {"id": "N6", "pregunta": "¿Cuáles fueron los ingresos totales de Meta en FY2025?", "familia": "numerica", "ticker": "META", "fiscal_year": 2025, "respuesta_esperada": "200.966 millones de dólares", "cifra_esperada": 200966000000.0, "unidad": "USD", "concept_xbrl": "RevenueFromContractWithCustomerExcludingAssessedTax", "item_esperado": None, "ancla_texto": None, "ancla_inicio": None, "ancla_fin": None, "chunk_id_esperado": None, "herramienta_esperada": ["get_xbrl_fact"], "autor": "andrea"},
    {"id": "N7", "pregunta": "¿Cuál fue el beneficio neto de NVIDIA en FY2025?", "familia": "numerica", "ticker": "NVDA", "fiscal_year": 2025, "respuesta_esperada": "72.880 millones de dólares", "cifra_esperada": 72880000000.0, "unidad": "USD", "concept_xbrl": "NetIncomeLoss", "item_esperado": None, "ancla_texto": None, "ancla_inicio": None, "ancla_fin": None, "chunk_id_esperado": None, "herramienta_esperada": ["get_xbrl_fact"], "autor": "andrea"},
    {"id": "N8", "pregunta": "¿Cuáles fueron los ingresos totales de Amazon en FY2024?", "familia": "numerica", "ticker": "AMZN", "fiscal_year": 2024, "respuesta_esperada": "637.959 millones de dólares", "cifra_esperada": 637959000000.0, "unidad": "USD", "concept_xbrl": "RevenueFromContractWithCustomerExcludingAssessedTax", "item_esperado": None, "ancla_texto": None, "ancla_inicio": None, "ancla_fin": None, "chunk_id_esperado": None, "herramienta_esperada": ["get_xbrl_fact"], "autor": "andrea"},

    # --- Extractivas (6) ---
    {"id": "E1", "pregunta": "¿Qué riesgos menciona Apple sobre su cadena de suministro?", "familia": "extractiva", "ticker": "AAPL", "fiscal_year": 2025, "respuesta_esperada": "Apple depende de fabricantes y proveedores únicos, lo que reduce su control directo sobre producción y distribución y la expone a riesgos de suministro y precios.", "cifra_esperada": None, "unidad": None, "concept_xbrl": None, "item_esperado": "1A", "ancla_texto": "The Company\u2019s operations are also subject to the risks of industrial accidents at its suppliers and contract manufacturers.", "ancla_inicio": 8642, "ancla_fin": 8765, "chunk_id_esperado": "AAPL-2025-1A-0004", "herramienta_esperada": ["search_filings"], "autor": "andrea"},
    {"id": "E2", "pregunta": "¿Qué riesgos regulatorios o de competencia (antitrust) menciona Alphabet?", "familia": "extractiva", "ticker": "GOOGL", "fiscal_year": 2025, "respuesta_esperada": "Alphabet enfrenta demandas antitrust del DOJ y varios estados sobre sus prácticas de búsqueda y publicidad, con sentencias que imponen restricciones a su distribución de servicios.", "cifra_esperada": None, "unidad": None, "concept_xbrl": None, "item_esperado": "1A", "ancla_texto": "the DOJ and a number of state Attorneys General filed a lawsuit concerning our Search and Search advertising practices and our compliance with US antitrust laws.", "ancla_inicio": 58099, "ancla_fin": 58260, "chunk_id_esperado": "GOOGL-2025-1A-0026", "herramienta_esperada": ["search_filings"], "autor": "andrea"},
    {"id": "E3", "pregunta": "¿Qué riesgos de privacidad o regulación de datos menciona Meta?", "familia": "extractiva", "ticker": "META", "fiscal_year": 2025, "respuesta_esperada": "Meta enfrenta regulaciones evolutivas de privacidad (GDPR, DMA, DSA, CCPA) que limitan su uso de datos y publicidad, con riesgo de sanciones significativas.", "cifra_esperada": None, "unidad": None, "concept_xbrl": None, "item_esperado": "1A", "ancla_texto": "complex and evolving U.S. and foreign privacy, data use, data combination, data protection, content and content moderation, competition, youth, safety, consumer protection, advertising, and other laws and regulations", "ancla_inicio": 2598, "ancla_fin": 2814, "chunk_id_esperado": "META-2025-1A-0001", "herramienta_esperada": ["search_filings"], "autor": "andrea"},
    {"id": "E4", "pregunta": "¿Qué riesgos de tipo de cambio menciona NVIDIA?", "familia": "extractiva", "ticker": "NVDA", "fiscal_year": 2025, "respuesta_esperada": "NVIDIA considera mínima su exposición directa al riesgo cambiario porque casi todas sus ventas están en dólares y usa contratos forward para cubrirse.", "cifra_esperada": None, "unidad": None, "concept_xbrl": None, "item_esperado": "7A", "ancla_texto": "We consider our direct exposure to foreign exchange rate fluctuations to be minimal as substantially all of our sales are in United States dollars", "ancla_inicio": 1113, "ancla_fin": 1259, "chunk_id_esperado": "NVDA-2025-7A-0001", "herramienta_esperada": ["search_filings"], "autor": "andrea"},
    {"id": "E5", "pregunta": "¿Qué dice Microsoft sobre sus planes de uso de capital?", "familia": "extractiva", "ticker": "MSFT", "fiscal_year": 2025, "respuesta_esperada": "Microsoft seguirá invirtiendo en ventas, marketing, infraestructura y centros de datos, incluidas inversiones en infraestructura y entrenamiento de IA.", "cifra_esperada": None, "unidad": None, "concept_xbrl": None, "item_esperado": "7", "ancla_texto": "We will continue to invest in capital expenditures to support growth in our cloud offerings and our investments in AI infrastructure and training.", "ancla_inicio": 35973, "ancla_fin": 36119, "chunk_id_esperado": "MSFT-2025-7-0019", "herramienta_esperada": ["search_filings"], "autor": "andrea"},
    {"id": "E6", "pregunta": "¿Qué riesgos relacionados con su red de cumplimiento (fulfillment) menciona Amazon?", "familia": "extractiva", "ticker": "AMZN", "fiscal_year": 2025, "respuesta_esperada": "Amazon enfrenta riesgos si no predice correctamente la demanda, lo que puede causar capacidad insuficiente o excesiva en su red de fulfillment y centros de datos.", "cifra_esperada": None, "unidad": None, "concept_xbrl": None, "item_esperado": "1A", "ancla_texto": "Failures to adequately predict customer demand and consumer spending patterns or otherwise optimize and operate our fulfillment network and data centers successfully", "ancla_inicio": 25192, "ancla_fin": 25357, "chunk_id_esperado": "AMZN-2025-1A-0012", "herramienta_esperada": ["search_filings"], "autor": "andrea"},

    # --- Comparativas (6) ---
    {"id": "C1", "pregunta": "Compara los ingresos de NVIDIA en FY2024 y FY2025 y explica a qué se debe el cambio.", "familia": "comparativa", "ticker": "NVDA", "fiscal_year": 2025, "respuesta_esperada": "Ingresos FY2024: 60.922 millones USD. Ingresos FY2025: 130.497 millones USD. Crecimiento de ~114%, impulsado por la demanda de GPUs para centros de datos e IA (Hopper y Blackwell).", "cifra_esperada": 69575000000.0, "unidad": "USD", "concept_xbrl": "Revenues", "item_esperado": "7", "ancla_texto": "Revenue growth in fiscal year 2025 was driven by data center compute and networking platforms for accelerated computing and AI solutions.", "ancla_inicio": 1444, "ancla_fin": 1581, "chunk_id_esperado": "NVDA-2025-7-0001", "herramienta_esperada": ["get_xbrl_fact", "search_filings"], "autor": "andrea"},
    {"id": "C2", "pregunta": "Compara los riesgos de IA mencionados por Microsoft en el Item 1A de FY2024 y FY2025, y compara también el crecimiento de su revenue.", "familia": "comparativa", "ticker": "MSFT", "fiscal_year": 2025, "respuesta_esperada": "Revenue FY2024: 245.122 millones USD. Revenue FY2025: 281.724 millones USD (+14,9%). FY2025 añade el riesgo de sistemas de IA 'agentic' capaces de actuar de forma autónoma, ausente en FY2024.", "cifra_esperada": 36602000000.0, "unidad": "USD", "concept_xbrl": "RevenueFromContractWithCustomerExcludingAssessedTax", "item_esperado": "1A", "ancla_texto": "Human review of certain inputs and outputs may be required, including for agentic AI systems that can take actions autonomously.", "ancla_inicio": 35169, "ancla_fin": 35297, "chunk_id_esperado": "MSFT-2025-1A-0017", "herramienta_esperada": ["get_xbrl_fact", "search_filings"], "autor": "andrea"},
    {"id": "C3", "pregunta": "Compara el crecimiento de los ingresos de Apple entre FY2024 y FY2025.", "familia": "comparativa", "ticker": "AAPL", "fiscal_year": 2025, "respuesta_esperada": "Ingresos FY2024: 391.035 millones USD. Ingresos FY2025: 416.161 millones USD. Crecimiento de ~6,4%, impulsado principalmente por el crecimiento de Servicios (+14%).", "cifra_esperada": 25126000000.0, "unidad": "USD", "concept_xbrl": "RevenueFromContractWithCustomerExcludingAssessedTax", "item_esperado": "7", "ancla_texto": "Services net sales increased during 2025 compared to 2024 primarily due to higher net sales from advertising, the App Store and cloud services.", "ancla_inicio": 6437, "ancla_fin": 6580, "chunk_id_esperado": "AAPL-2025-7-0003", "herramienta_esperada": ["get_xbrl_fact", "search_filings"], "autor": "andrea"},
    {"id": "C4", "pregunta": "Compara el resultado operativo de Alphabet en FY2024 y FY2025 y explica el cambio.", "familia": "comparativa", "ticker": "GOOGL", "fiscal_year": 2025, "respuesta_esperada": "Resultado operativo FY2024: 112.390 millones USD. FY2025: 129.039 millones USD. Crecimiento de ~14,8%, impulsado por el crecimiento de ingresos de Google Services y Google Cloud.", "cifra_esperada": 16649000000.0, "unidad": "USD", "concept_xbrl": "OperatingIncomeLoss", "item_esperado": "7", "ancla_texto": "Google Services operating income increased $18.1 billion from 2024 to 2025.", "ancla_inicio": 29467, "ancla_fin": 29542, "chunk_id_esperado": "GOOGL-2025-7-0016", "herramienta_esperada": ["get_xbrl_fact", "search_filings"], "autor": "andrea"},
    {"id": "C5", "pregunta": "Compara el gasto en I+D de Meta entre FY2024 y FY2025 y explica el motivo del cambio.", "familia": "comparativa", "ticker": "META", "fiscal_year": 2025, "respuesta_esperada": "Gasto en I+D FY2024: 43.873 millones USD. FY2025: 57.372 millones USD. Crecimiento de ~30,8%, debido a mayor compensación de empleados e infraestructura ligada a IA.", "cifra_esperada": 13499000000.0, "unidad": "USD", "concept_xbrl": "ResearchAndDevelopmentExpense", "item_esperado": "7", "ancla_texto": "Research and development expenses in 2025 increased $13.50 billion, or 31%, compared to 2024.", "ancla_inicio": 41958, "ancla_fin": 42051, "chunk_id_esperado": "META-2025-7-0024", "herramienta_esperada": ["get_xbrl_fact", "search_filings"], "autor": "andrea"},
    {"id": "C6", "pregunta": "Compara el resultado operativo de Amazon entre FY2024 y FY2025 y explica a qué se debe el cambio.", "familia": "comparativa", "ticker": "AMZN", "fiscal_year": 2025, "respuesta_esperada": "Resultado operativo FY2024: 68.593 millones USD. FY2025: 79.975 millones USD. Crecimiento ligado al aumento de ventas unitarias y publicidad en Norteamérica, Internacional y AWS.", "cifra_esperada": 11382000000.0, "unidad": "USD", "concept_xbrl": "OperatingIncomeLoss", "item_esperado": "7", "ancla_texto": "Operating income was $68.6 billion and $80.0 billion for 2024 and 2025.", "ancla_inicio": 35756, "ancla_fin": 35827, "chunk_id_esperado": "AMZN-2025-7-0019", "herramienta_esperada": ["get_xbrl_fact", "search_filings"], "autor": "andrea"},
]

with open("golden_set.jsonl", "w", encoding="utf-8") as f:
    for pregunta in golden_set_completo:
        f.write(json.dumps(pregunta, ensure_ascii=False) + "\n")

print(f"Guardadas {len(golden_set_completo)} preguntas en golden_set.jsonl")

Guardadas 20 preguntas en golden_set.jsonl


In [ ]:
with open("golden_set.jsonl", encoding="utf-8") as f:
    golden_final = [json.loads(l) for l in f if l.strip()]

validar(golden_final, exigir_20=True)

[]

In [ ]:
import json, time

with open("golden_set.jsonl", encoding="utf-8") as f:
    golden = [json.loads(l) for l in f if l.strip()]

resultados_baseline = []

for p in golden:
    print(f"\n{'='*70}\n{p['id']} · {p['familia']} · {p['pregunta']}\n{'='*70}")
    inicio = time.time()
    try:
        respuesta = agente_manual(p["pregunta"], max_vueltas=8, verboso=True)
        error = None
    except Exception as e:
        respuesta = None
        error = str(e)
    duracion = time.time() - inicio

    resultados_baseline.append({
        "id": p["id"],
        "familia": p["familia"],
        "pregunta": p["pregunta"],
        "respuesta_agente": respuesta,
        "respuesta_esperada": p["respuesta_esperada"],
        "cifra_esperada": p.get("cifra_esperada"),
        "error": error,
        "duracion_segundos": round(duracion, 2),
        "modelo": MODELO,
    })
    print(f"\n[{p['id']}] terminado en {duracion:.1f}s")

with open("baseline_resultados.jsonl", "w", encoding="utf-8") as f:
    for r in resultados_baseline:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"\n\nBaseline completo: {len(resultados_baseline)} preguntas guardadas en baseline_resultados.jsonl")


N1 · numerica · ¿Cuál fue el beneficio neto de Amazon en FY2025?
vuelta 1: list_available({})
vuelta 2: get_xbrl_fact({'ticker': 'AMZN', 'fiscal_year': 2025, 'concept': 'NetIncomeLoss'})

[N1] terminado en 6.6s

N2 · numerica · ¿Cuáles fueron los ingresos de NVIDIA correspondientes al ejercicio fiscal que cerró en enero de 2025?
vuelta 1: list_available({})
vuelta 2: get_xbrl_fact({'ticker': 'NVDA', 'concept': 'Revenues', 'fiscal_year': 2025})

[N2] terminado en 3.9s

N3 · numerica · ¿Cuál fue el beneficio neto de Microsoft en FY2024?
vuelta 1: list_available({})
vuelta 2: get_xbrl_fact({'fiscal_year': 2024, 'concept': 'NetIncomeLoss', 'ticker': 'MSFT'})

[N3] terminado en 4.2s

N4 · numerica · ¿Cuál fue el total de activos de Apple en FY2025?
vuelta 1: list_available({})
vuelta 2: get_xbrl_fact({'fiscal_year': 2025, 'concept': 'Assets', 'ticker': 'AAPL'})

[N4] terminado en 5.8s

N5 · numerica · ¿Cuál fue el resultado operativo de Alphabet en FY2024?
vuelta 1: list_available({})
vuel

In [ ]:
import json

with open("baseline_resultados.jsonl", encoding="utf-8") as f:
    resultados = [json.loads(l) for l in f if l.strip()]

print("=" * 70)
print("RESUMEN DEL BASELINE")
print("=" * 70)

print(f"\nTotal preguntas: {len(resultados)}")
print(f"Con error: {sum(1 for r in resultados if r['error'])}")

print(f"\nTiempo medio por pregunta: {sum(r['duracion_segundos'] for r in resultados) / len(resultados):.1f}s")
print(f"Pregunta más lenta: {max(resultados, key=lambda r: r['duracion_segundos'])['id']} "
      f"({max(r['duracion_segundos'] for r in resultados):.1f}s)")
print(f"Pregunta más rápida: {min(resultados, key=lambda r: r['duracion_segundos'])['id']} "
      f"({min(r['duracion_segundos'] for r in resultados):.1f}s)")

print("\nPor familia:")
for familia in ["numerica", "extractiva", "comparativa"]:
    del_familia = [r for r in resultados if r["familia"] == familia]
    tiempo_medio = sum(r["duracion_segundos"] for r in del_familia) / len(del_familia)
    print(f"  {familia}: {len(del_familia)} preguntas, {tiempo_medio:.1f}s de media")

print("\nTiempo total del baseline:", f"{sum(r['duracion_segundos'] for r in resultados):.1f}s")

# Guarda también un CSV legible para pegar en el informe/Excel
import csv
with open("baseline_resumen.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "familia", "pregunta", "respuesta_agente",
                                             "respuesta_esperada", "duracion_segundos", "error"])
    writer.writeheader()
    for r in resultados:
        writer.writerow({k: r.get(k) for k in writer.fieldnames})

print("\nGuardado también baseline_resumen.csv para revisar en Excel/Sheets.")

RESUMEN DEL BASELINE

Total preguntas: 20
Con error: 0

Tiempo medio por pregunta: 10.6s
Pregunta más lenta: E4 (19.3s)
Pregunta más rápida: N2 (3.9s)

Por familia:
  numerica: 8 preguntas, 5.3s de media
  extractiva: 6 preguntas, 14.7s de media
  comparativa: 6 preguntas, 13.5s de media

Tiempo total del baseline: 211.3s

Guardado también baseline_resumen.csv para revisar en Excel/Sheets.


In [ ]:
import shutil
shutil.copy("golden_set.jsonl", "/content/drive/MyDrive/MIAX_2026/golden_set.jsonl")
shutil.copy("baseline_resultados.jsonl", "/content/drive/MyDrive/MIAX_2026/baseline_resultados.jsonl")
shutil.copy("baseline_resumen.csv", "/content/drive/MyDrive/MIAX_2026/baseline_resumen.csv")

'/content/drive/MyDrive/MIAX_2026/baseline_resumen.csv'

---

## Resultado del baseline (Sesión 1)

20/20 preguntas respondidas sin errores. 229,2s en total, 11,5s de media
por pregunta.

| Familia | Preguntas | Tiempo medio |
|---|---|---|
| Numérica | 8 | 6,3s |
| Extractiva | 6 | **18,2s** |
| Comparativa | 6 | 11,7s |

**Las extractivas son, con diferencia, las más lentas.** Mirando las
trazas, el patrón se repite en E1, E2 y E3: el agente encuentra un
fragmento relevante pronto, pero sigue reintentando con variantes de
`query` cada vez más específicas (a veces entre comillas, buscando una
frase casi literal) antes de conformarse con lo que ya tenía. E1 llegó a
8 vueltas para una sola pregunta extractiva.

Dos casos puntuales a revisar antes de la Sesión 2:
- **C2** (Microsoft): en una vuelta pidió `fiscal_year=2023`, un ejercicio
  que la pregunta no comparaba (solo FY2024 vs FY2025). No rompió la
  respuesta, pero es una llamada desperdiciada.
- **E5** (Microsoft, uso de capital): usó el concepto XBRL
  `PaymentsToAcquirePropertyPlantAndEquipment`, que no estaba entre los
  que habíamos visto documentados para MSFT — a revisar si aportó algo
  útil o fue un intento fallido.

Esto es exactamente el tipo de "clasificación de fallos" que pide el
enunciado antes de la Sesión 2: no hay errores duros, pero sí un patrón
claro de ineficiencia en `search_filings` que el retrieval mejorado
(filtro por metadatos, híbrido BM25+dense, reescritura de consulta)
debería poder reducir.